In [ ]:
from io import StringIO
import importlib.util
import json
import os
import platform
import py_compile
import shutil
import sys
import tarfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({'figure.dpi': 130, 'savefig.dpi': 160, 'font.size': 9, 'axes.titleweight': 'bold', 'axes.spines.top': False, 'axes.spines.right': False})
DATA = json.loads('{"snapshot_date": "2026-06-28", "assembled_on": "2026-06-29", "default_selection": "A", "decision_mode": "CHALLENGER_BY_USER_APPROVAL", "field_chart_csv": "archetype,games,usage_share,usage_pct,score_rate,score_pct,wilson_low,wilson_high\\nalakazam_dunsparce,2244,0.189559,18.9559,0.513369,51.3369,0.492683,0.534009\\nlucario,2223,0.187785,18.7785,0.424426,42.442600000000006,0.404028,0.445086\\nhop_trevenant,2074,0.175199,17.5199,0.454677,45.4677,0.43335,0.476171\\narchaludon,1725,0.145717,14.571700000000002,0.622029,62.20290000000001,0.5989,0.644616\\nstarmie,1640,0.138537,13.8537,0.518902,51.8902,0.494704,0.543012\\ndragapult,865,0.07307,7.3069999999999995,0.491329,49.1329,0.458126,0.52461\\niono_kilowattrel,152,0.01284,1.284,0.348684,34.8684,0.2775,0.427329\\nteam_rocket_spidops,145,0.012249,1.2248999999999999,0.641379,64.1379,0.560595,0.714866\\nfestival_thwackey,137,0.011573,1.1573,0.459854,45.9854,0.378631,0.543267\\nother:Team Rocket\'s Honchkrow|Team Rocket\'s Murkrow,111,0.009377,0.9377,0.581081,58.10809999999999,0.48809,0.668648\\n", "trend_chart_csv": "archetype,usage_delta,usage_delta_pct,usage_0627,usage_0627_pct,score_0627,score_0627_pct,score_delta,games_0627\\narchaludon,0.058753,5.8753,0.145717,14.571700000000002,0.622029,62.20290000000001,-0.0217079999999999,1725\\nstarmie,0.023894,2.3893999999999997,0.138537,13.8537,0.518902,51.8902,-0.022765,1640\\nother:Team Rocket\'s Honchkrow|Team Rocket\'s Murkrow,0.004198,0.41980000000000006,0.009377,0.9377,0.581081,58.10809999999999,0.0983219999999999,111\\nother:Cynthia\'s Gabite|Cynthia\'s Gible,0.003529,0.3529,0.003886,0.3886,0.543478,54.3478,0.043478,46\\nalakazam_dunsparce,0.002952,0.2952,0.189559,18.9559,0.513369,51.3369,-0.000746,2244\\nother:Okidogi[ID:116]|Solrock,0.002742,0.2742,0.008278,0.8278000000000001,0.612245,61.224500000000006,0.047729,98\\nother:Teal Mask Ogerpon ex|Raging Bolt ex,0.00245,0.245,0.00245,0.245,0.586207,58.62070000000001,,29\\nother:Okidogi[ID:116]|Binacle,0.001858,0.1858,0.001858,0.1858,0.409091,40.909099999999995,,22\\nother:Handheld Fan|Turtonator[ID:1027],0.001183,0.1183,0.001183,0.1183,0.642857,64.2857,,14\\nother:Walrein|Spheal,0.0005139999999999,0.051399999999989995,0.005068,0.5067999999999999,0.466667,46.6667,0.015687,60\\nother:Marill[ID:961]|Azumarill,0.0002689999999999,0.02689999999999,0.003126,0.3126,0.324324,32.4324,-0.175676,37\\nother:Snorunt[ID:860]|Mega Froslass ex,3.199999999999991e-05,0.003199999999999991,0.001014,0.10139999999999999,0.666667,66.6667,0.212122,12\\n", "ev_chart_csv": "archetype,usage_share,usage_pct,field_score_rate,field_score_pct,matchup_weighted_winrate,ev_pct,coverage_ratio,coverage_pct,n_covered_matchups,favorable_matchups,hard_matchups\\nteam_rocket_spidops,0.012249,1.2248999999999999,0.641379,64.1379,0.711997663539677,71.1997663539677,0.6996500130093516,69.96500130093516,4,alakazam_dunsparce:1.000(n=31); lucario:0.708(n=24); hop_trevenant:0.714(n=21),starmie:0.320(n=25)\\nother:Okidogi[ID:116]|Solrock,0.008278,0.8278000000000001,0.612245,61.224500000000006,0.7019739978548842,70.19739978548843,0.3163547849094807,31.63547849094807,2,hop_trevenant:0.783(n=23); starmie:0.600(n=20),\\nother:Team Rocket\'s Honchkrow|Team Rocket\'s Murkrow,0.009377,0.9377,0.581081,58.10809999999999,0.6578121368291698,65.78121368291698,0.5577732396683703,55.77732396683703,3,alakazam_dunsparce:0.955(n=22); hop_trevenant:0.625(n=28),lucario:0.389(n=36)\\narchaludon,0.145717,14.571700000000002,0.622029,62.20290000000001,0.6386960386348333,63.86960386348333,0.923069989687258,92.3069989687258,7,lucario:0.644(n=275); hop_trevenant:0.806(n=279); starmie:0.738(n=275); dragapult:0.662(n=154); iono_kilowattrel:0.640(n=25),alakazam_dunsparce:0.391(n=327)\\nstarmie,0.138537,13.8537,0.518902,51.8902,0.5314006193906129,53.14006193906129,0.9326343673494972,93.26343673494972,8,hop_trevenant:0.794(n=262); dragapult:0.698(n=129); team_rocket_spidops:0.680(n=25); festival_thwackey:0.714(n=21),lucario:0.395(n=291); archaludon:0.262(n=275); other:Okidogi[ID:116]|Solrock:0.400(n=20)\\nalakazam_dunsparce,0.189559,18.9559,0.513369,51.3369,0.517623241600737,51.7623241600737,0.9455925847779172,94.55925847779172,9,lucario:0.629(n=437); archaludon:0.609(n=327); iono_kilowattrel:0.667(n=36),team_rocket_spidops:0.000(n=31); festival_thwackey:0.387(n=31); other:Team Rocket\'s Honchkrow|Team Rocket\'s Murkrow:0.045(n=22)\\ndragapult,0.07307,7.3069999999999995,0.491329,49.1329,0.4947237828039537,49.47237828039537,0.9027618050985512,90.27618050985512,5,hop_trevenant:0.848(n=151),archaludon:0.338(n=154); starmie:0.302(n=129)\\nfestival_thwackey,0.011573,1.1573,0.459854,45.9854,0.4412063424636216,44.12063424636216,0.6566109586241574,65.66109586241573,4,alakazam_dunsparce:0.613(n=31),archaludon:0.273(n=22); starmie:0.286(n=21)\\nhop_trevenant,0.175199,17.5199,0.454677,45.4677,0.4312055557013126,43.12055557013126,0.9565761923179044,95.65761923179043,10,lucario:0.773(n=434); iono_kilowattrel:0.680(n=25),archaludon:0.194(n=279); starmie:0.206(n=262); dragapult:0.152(n=151); team_rocket_spidops:0.286(n=21); other:Team Rocket\'s Honchkrow|Team Rocket\'s Murkrow:0.375(n=28)\\nlucario,0.187785,18.7785,0.424426,42.442600000000006,0.4086412339560742,40.86412339560742,0.9314627284647536,93.14627284647537,8,starmie:0.605(n=291); iono_kilowattrel:0.833(n=30); other:Team Rocket\'s Honchkrow|Team Rocket\'s Murkrow:0.611(n=36),alakazam_dunsparce:0.371(n=437); hop_trevenant:0.227(n=434); archaludon:0.356(n=275); team_rocket_spidops:0.292(n=24)\\niono_kilowattrel,0.01284,1.284,0.348684,34.8684,0.2907307202789791,29.07307202789791,0.7073422748085416,70.73422748085416,4,,alakazam_dunsparce:0.333(n=36); lucario:0.167(n=30); hop_trevenant:0.320(n=25); archaludon:0.360(n=25)\\nother:Snorunt[ID:860]|Mega Froslass ex,0.001014,0.10139999999999999,0.666667,66.6667,,,0.0,0.0,0,,\\n", "matchup_csv": "", "strategy_csv": "archetype,n_games,class,first_attack_turn_mean,attack_cadence_mean,prize_rate,abilities_per_turn,draw_search_per_turn,top1_attack_share,attack_entropy,setup_actions_before_first_attack_mean\\nalakazam_dunsparce,1152,combo,4.003,0.4931,0.288,0.9757,4.6561,0.8954,0.1878,27.392\\narchaludon,1073,aggro,2.659,0.6755,0.3226,0.2519,5.3534,0.6527,0.7275,14.241\\nlucario,943,aggro,2.91,0.6105,0.3231,0.3275,6.7239,0.4755,0.5832,22.402\\nhop_trevenant,941,midrange,3.297,0.5599,0.2585,0.4406,4.0986,0.4712,0.507,19.091\\nstarmie,851,aggro,2.871,0.6488,0.4045,0.2175,4.0044,0.6503,0.5102,13.444\\ndragapult,425,midrange,2.873,0.6221,0.3984,0.5571,8.0955,0.6842,0.3993,18.219\\nteam_rocket_spidops,93,midrange,5.11,0.4558,0.2879,0.2614,4.0996,0.5136,0.5735,28.538\\n", "screen_candidates_csv": "candidate_id,candidate_archetype,covered_archetypes,covered_field_weight,validated_covered_archetypes,validated_covered_field_weight,local_games,weighted_score_rate,validated_weighted_score_rate,unweighted_score_rate,weighted_edge_vs_field,validated_weighted_edge_vs_field,worst_weighted_matchup,unvalidated_archetypes,notes\\nflex_archaludon_0018_minus1182_plus1213,archaludon,5,0.844302,5,0.844302,80,0.754687,0.754687,0.8125,0.182869,0.182869,archaludon,,field-weighted over covered opponent archetypes; validated_* columns exclude archetypes without fidelity-passed pilots\\nflex_archaludon_0001_minus1122_plus8,archaludon,5,0.844302,5,0.844302,80,0.732338,0.732338,0.775,0.160521,0.160521,archaludon,,field-weighted over covered opponent archetypes; validated_* columns exclude archetypes without fidelity-passed pilots\\nflex_alakazam_dunsparce_0000_seed,alakazam_dunsparce,5,0.844302,5,0.844302,80,0.726061,0.726061,0.75,0.180468,0.180468,archaludon,,field-weighted over covered opponent archetypes; validated_* columns exclude archetypes without fidelity-passed pilots\\nflex_alakazam_dunsparce_0002_minus5_plus1266,alakazam_dunsparce,5,0.844302,5,0.844302,80,0.678503,0.678503,0.7,0.13291,0.13291,lucario,,field-weighted over covered opponent archetypes; validated_* columns exclude archetypes without fidelity-passed pilots\\npublic_alakazam_5th,alakazam_dunsparce,5,0.844302,5,0.844302,80,0.665343,0.665343,0.6625,0.11975,0.11975,alakazam_dunsparce,,field-weighted over covered opponent archetypes; validated_* columns exclude archetypes without fidelity-passed pilots\\npublic915_search_lucario,lucario,5,0.844302,5,0.844302,80,0.578469,0.578469,0.63125,0.196272,0.196272,alakazam_dunsparce,,field-weighted over covered opponent archetypes; validated_* columns exclude archetypes without fidelity-passed pilots\\nmeta_snapshot_930_submission_b,lucario,5,0.844302,5,0.844302,80,0.570757,0.570757,0.65,0.188561,0.188561,archaludon,,field-weighted over covered opponent archetypes; validated_* columns exclude archetypes without fidelity-passed pilots\\npublic1084_lucario,lucario,5,0.844302,5,0.844302,80,0.489294,0.489294,0.575,0.107098,0.107098,archaludon,,field-weighted over covered opponent archetypes; validated_* columns exclude archetypes without fidelity-passed pilots\\n", "holdout_candidates_csv": "candidate_id,candidate_archetype,covered_archetypes,covered_field_weight,validated_covered_archetypes,validated_covered_field_weight,local_games,weighted_score_rate,validated_weighted_score_rate,unweighted_score_rate,weighted_edge_vs_field,validated_weighted_edge_vs_field,worst_weighted_matchup,unvalidated_archetypes,notes\\nflex_alakazam_dunsparce_0000_seed,alakazam_dunsparce,5,0.844302,5,0.844302,200,0.710774,0.710774,0.75,0.165182,0.165182,archaludon,,field-weighted over covered opponent archetypes; validated_* columns exclude archetypes without fidelity-passed pilots\\nflex_archaludon_0018_minus1182_plus1213,archaludon,5,0.844302,5,0.844302,200,0.708502,0.708502,0.785,0.136685,0.136685,archaludon,,field-weighted over covered opponent archetypes; validated_* columns exclude archetypes without fidelity-passed pilots\\nflex_alakazam_dunsparce_0002_minus5_plus1266,alakazam_dunsparce,5,0.844302,5,0.844302,200,0.692496,0.692496,0.745,0.146903,0.146903,archaludon,,field-weighted over covered opponent archetypes; validated_* columns exclude archetypes without fidelity-passed pilots\\nflex_archaludon_0001_minus1122_plus8,archaludon,5,0.844302,5,0.844302,200,0.645367,0.645367,0.73,0.07355,0.07355,archaludon,,field-weighted over covered opponent archetypes; validated_* columns exclude archetypes without fidelity-passed pilots\\npublic_alakazam_5th,alakazam_dunsparce,5,0.844302,5,0.844302,200,0.527565,0.527565,0.585,-0.018028,-0.018028,archaludon,,field-weighted over covered opponent archetypes; validated_* columns exclude archetypes without fidelity-passed pilots\\n", "debias_csv": "candidate_id,candidate_archetype,selection_score,selection_edge,selection_games,holdout_score,holdout_edge,holdout_games,score_drop,edge_drop,search_n,selection_se,winner_curse_penalty,candidate_mean_score,shrinkage_factor,shrunk_score,shrunk_edge,publish_score,publish_edge,debias_method,debias_status,reason\\nflex_alakazam_dunsparce_0000_seed,alakazam_dunsparce,0.726061,0.180468,80,0.710774,0.165182,200.0,0.015287,0.015286,30,0.049862,0.130047,0.552649,0.861342,0.702016,0.155445,0.710774,0.165182,holdout,holdout_pass,passes de-biased score and edge bars\\nflex_archaludon_0018_minus1182_plus1213,archaludon,0.754687,0.182869,80,0.708502,0.136685,200.0,0.046185,0.046184,30,0.048106,0.125467,0.552649,0.861342,0.726673,0.157513,0.708502,0.136685,holdout,holdout_pass,passes de-biased score and edge bars\\nflex_alakazam_dunsparce_0002_minus5_plus1266,alakazam_dunsparce,0.678503,0.13291,80,0.692496,0.146903,200.0,-0.013993,-0.013993,30,0.052218,0.136192,0.552649,0.861342,0.661052,0.114481,0.692496,0.146903,holdout,holdout_pass,passes de-biased score and edge bars\\nflex_archaludon_0001_minus1122_plus8,archaludon,0.732338,0.160521,80,0.645367,0.07355,200.0,0.086971,0.086971,30,0.0495,0.129103,0.552649,0.861342,0.707423,0.138263,0.645367,0.07355,holdout,holdout_pass,passes de-biased score and edge bars\\npublic_alakazam_5th,alakazam_dunsparce,0.665343,0.11975,80,0.527565,-0.018028,200.0,0.137778,0.137778,30,0.052757,0.137597,0.552649,0.861342,0.649717,0.103146,0.527565,-0.018028,holdout,holdout_fail,publish_edge<0.030\\npublic915_search_lucario,lucario,0.578469,0.196272,80,,,,,,30,0.055209,0.143993,0.552649,0.861342,0.574889,0.169057,0.430896,0.169057,shrunk,shrunk_fail,publish_score<0.520\\nmeta_snapshot_930_submission_b,lucario,0.570757,0.188561,80,,,,,,30,0.055339,0.144332,0.552649,0.861342,0.568246,0.162415,0.423914,0.162415,shrunk,shrunk_fail,publish_score<0.520\\npublic1084_lucario,lucario,0.489294,0.107098,80,,,,,,30,0.055889,0.145766,0.552649,0.861342,0.498079,0.092248,0.352313,0.092248,shrunk,shrunk_fail,publish_score<0.520\\n", "gap_csv": "candidate_id,candidate_archetype,local_weighted_score,local_weighted_edge,covered_field_weight,unweighted_score,prior_reference_score,early_live_score,live_delta_points,live_gap_status,live_reference_gate_decision,a1_panel_fidelity_status,a2_coverage_status,a3_weighting_status,b1_supremacy_status,c2_debias_status,c3_drift_status,b2_runtime_status,dominant_causes,gap_clean,recommended_route,notes\\nflex_archaludon_0018_minus1182_plus1213,archaludon,0.754687,0.182869,0.844302,0.8125,,,,live_unknown,PROVISIONAL_MORE_DATA,pass,pass,pass,supremacy_pass,holdout_pass,not_checked,pass,,True,no_gap_action_required,\\"diagnose before pilot repair, deck tuning, or submit-capable snapshot promotion\\"\\nflex_archaludon_0001_minus1122_plus8,archaludon,0.732338,0.160521,0.844302,0.775,,,,live_unknown,PROVISIONAL_MORE_DATA,pass,pass,pass,supremacy_pass,holdout_pass,not_checked,pass,,True,no_gap_action_required,\\"diagnose before pilot repair, deck tuning, or submit-capable snapshot promotion\\"\\nflex_alakazam_dunsparce_0000_seed,alakazam_dunsparce,0.726061,0.180468,0.844302,0.75,,,,live_unknown,PROVISIONAL_MORE_DATA,pass,pass,pass,supremacy_pass,holdout_pass,not_checked,pass,,True,no_gap_action_required,\\"diagnose before pilot repair, deck tuning, or submit-capable snapshot promotion\\"\\nflex_alakazam_dunsparce_0002_minus5_plus1266,alakazam_dunsparce,0.678503,0.13291,0.844302,0.7,,,,live_unknown,PROVISIONAL_MORE_DATA,pass,pass,pass,supremacy_pass,holdout_pass,not_checked,pass,,True,no_gap_action_required,\\"diagnose before pilot repair, deck tuning, or submit-capable snapshot promotion\\"\\npublic_alakazam_5th,alakazam_dunsparce,0.665343,0.11975,0.844302,0.6625,,,,live_unknown,PROVISIONAL_MORE_DATA,pass,pass,pass,supremacy_pass,holdout_fail,not_checked,pass,C2,False,fresh_holdout_or_shrinkage,\\"diagnose before pilot repair, deck tuning, or submit-capable snapshot promotion\\"\\npublic915_search_lucario,lucario,0.578469,0.196272,0.844302,0.63125,,,,live_unknown,PROVISIONAL_MORE_DATA,pass,pass,pass,supremacy_pass,shrunk_fail,not_checked,pass,C2,False,fresh_holdout_or_shrinkage,\\"diagnose before pilot repair, deck tuning, or submit-capable snapshot promotion\\"\\nmeta_snapshot_930_submission_b,lucario,0.570757,0.188561,0.844302,0.65,,,,live_unknown,PROVISIONAL_MORE_DATA,pass,pass,pass,supremacy_pass,shrunk_fail,not_checked,pass,C2,False,fresh_holdout_or_shrinkage,\\"diagnose before pilot repair, deck tuning, or submit-capable snapshot promotion\\"\\npublic1084_lucario,lucario,0.489294,0.107098,0.844302,0.575,930.6,804.0,-126.6,live_regression,HOLD_DO_NOT_SUBMIT,pass,pass,pass,supremacy_pass,shrunk_fail,not_checked,pass,C2,False,fresh_holdout_or_shrinkage,\\"diagnose before pilot repair, deck tuning, or submit-capable snapshot promotion\\"\\n", "hold_candidates_csv": "candidate_id,candidate_archetype,weighted_score_rate,weighted_edge_vs_field,covered_field_weight,wilson_low,runtime_risk,coverage_penalty,gap_penalty,debias_penalty,live_reference_penalty,robustness_penalty,held_score,known_reference_margin_points,gap_clean,debias_status,decision,reason\\nflex_archaludon_0018_minus1182_plus1213,archaludon,0.754687,0.182869,0.844302,0.713415,0.0,0.0,0.0,0.0,0.0,0.0,0.734051,0.0,True,holdout_pass,PROMOTE_CANDIDATE,passes hold-period gate\\nflex_archaludon_0001_minus1122_plus8,archaludon,0.732338,0.160521,0.844302,0.67213,0.0,0.0,0.0,0.0,0.0,0.0,0.702234,0.0,True,holdout_pass,PROMOTE_CANDIDATE,passes hold-period gate\\nflex_alakazam_dunsparce_0000_seed,alakazam_dunsparce,0.726061,0.180468,0.844302,0.645151,0.0,0.0,0.0,0.0,0.0,0.0,0.685606,0.0,True,holdout_pass,PROMOTE_CANDIDATE,passes hold-period gate\\nflex_alakazam_dunsparce_0002_minus5_plus1266,alakazam_dunsparce,0.678503,0.13291,0.844302,0.592316,0.0,0.0,0.0,0.0,0.0,0.0,0.635409,0.0,True,holdout_pass,PROMOTE_CANDIDATE,passes hold-period gate\\npublic_alakazam_5th,alakazam_dunsparce,0.665343,0.11975,0.844302,0.553563,0.0,0.0,0.08,0.06,0.0,0.14,0.525343,0.0,False,holdout_fail,HOLD_DO_NOT_SUBMIT,gap diagnosis is not clean; de-biased edge missing\\npublic915_search_lucario,lucario,0.578469,0.196272,0.844302,0.521787,0.0,0.0,0.08,0.06,0.0,0.14,0.438469,0.0,False,shrunk_fail,HOLD_DO_NOT_SUBMIT,gap diagnosis is not clean; de-biased edge missing; held score below promotion bar\\nmeta_snapshot_930_submission_b,lucario,0.570757,0.188561,0.844302,0.540798,0.0,0.0,0.08,0.06,0.0,0.14,0.430757,0.0,False,shrunk_fail,HOLD_DO_NOT_SUBMIT,gap diagnosis is not clean; de-biased edge missing; held score below promotion bar\\npublic1084_lucario,lucario,0.489294,0.107098,0.844302,0.465691,0.0,0.0,0.08,0.06,0.08,0.22,0.269294,-126.6,False,shrunk_fail,HOLD_DO_NOT_SUBMIT,gap diagnosis is not clean; de-biased edge missing; early live/reference regression exceeds guard; held score below promotion bar\\n", "pair_scores_csv": "portfolio_ids,candidate_a,candidate_b,candidate_a_archetype,candidate_b_archetype,same_archetype_pair,mean_best_score_rate,min_member_held_score,mean_member_held_score,unique_edge_a,unique_edge_b,both_below_50_opponents,pair_held_score,decision,reason\\nflex_archaludon_0018_minus1182_plus1213;flex_alakazam_dunsparce_0000_seed,flex_archaludon_0018_minus1182_plus1213,flex_alakazam_dunsparce_0000_seed,archaludon,alakazam_dunsparce,0,0.875,0.685606,0.709828,2,1,0,0.772424,PROMOTE_PAIR,passes pair hold-period gate\\nflex_archaludon_0001_minus1122_plus8;flex_alakazam_dunsparce_0000_seed,flex_archaludon_0001_minus1122_plus8,flex_alakazam_dunsparce_0000_seed,archaludon,alakazam_dunsparce,0,0.8375,0.685606,0.69392,1,1,0,0.751174,PROMOTE_PAIR,passes pair hold-period gate\\nflex_archaludon_0018_minus1182_plus1213;flex_alakazam_dunsparce_0002_minus5_plus1266,flex_archaludon_0018_minus1182_plus1213,flex_alakazam_dunsparce_0002_minus5_plus1266,archaludon,alakazam_dunsparce,0,0.825,0.635409,0.68473,3,0,0,0.722286,PROMOTE_PAIR,passes pair hold-period gate\\nflex_archaludon_0018_minus1182_plus1213;flex_alakazam_dunsparce_0006_minus5_plus1156,flex_archaludon_0018_minus1182_plus1213,flex_alakazam_dunsparce_0006_minus5_plus1156,archaludon,alakazam_dunsparce,0,0.825,0.620151,0.677101,2,0,0,0.701606,PROMOTE_PAIR,passes pair hold-period gate\\nflex_alakazam_dunsparce_0000_seed;flex_archaludon_0021_minus1182_plus1087,flex_alakazam_dunsparce_0000_seed,flex_archaludon_0021_minus1182_plus1087,alakazam_dunsparce,archaludon,0,0.7875,0.647263,0.666435,1,0,0,0.699334,PROMOTE_PAIR,passes pair hold-period gate\\nflex_archaludon_0001_minus1122_plus8;flex_alakazam_dunsparce_0002_minus5_plus1266,flex_archaludon_0001_minus1122_plus8,flex_alakazam_dunsparce_0002_minus5_plus1266,archaludon,alakazam_dunsparce,0,0.775,0.635409,0.668821,1,0,0,0.687286,PROMOTE_PAIR,passes pair hold-period gate\\nflex_archaludon_0021_minus1182_plus1087;flex_alakazam_dunsparce_0002_minus5_plus1266,flex_archaludon_0021_minus1182_plus1087,flex_alakazam_dunsparce_0002_minus5_plus1266,archaludon,alakazam_dunsparce,0,0.7625,0.635409,0.641336,1,0,0,0.683536,PROMOTE_PAIR,passes pair hold-period gate\\nflex_archaludon_0001_minus1122_plus8;flex_alakazam_dunsparce_0006_minus5_plus1156,flex_archaludon_0001_minus1122_plus8,flex_alakazam_dunsparce_0006_minus5_plus1156,archaludon,alakazam_dunsparce,0,0.775,0.620151,0.661193,1,0,0,0.676606,PROMOTE_PAIR,passes pair hold-period gate\\nflex_archaludon_0010_minus1147_plus1192;flex_alakazam_dunsparce_0000_seed,flex_archaludon_0010_minus1147_plus1192,flex_alakazam_dunsparce_0000_seed,archaludon,alakazam_dunsparce,0,0.9375,0.673559,0.679582,2,1,0,0.782741,HOLD_DO_NOT_SUBMIT,flex_archaludon_0010_minus1147_plus1192:HOLD_DO_NOT_SUBMIT\\nflex_archaludon_0018_minus1182_plus1213;flex_archaludon_0001_minus1122_plus8,flex_archaludon_0018_minus1182_plus1213,flex_archaludon_0001_minus1122_plus8,archaludon,archaludon,1,0.8375,0.702234,0.718143,1,0,0,0.752814,HOLD_DO_NOT_SUBMIT,same archetype pair lacks portfolio diversity\\nflex_archaludon_0010_minus1147_plus1192;flex_archaludon_0021_minus1182_plus1087,flex_archaludon_0010_minus1147_plus1192,flex_archaludon_0021_minus1182_plus1087,archaludon,archaludon,1,0.875,0.647263,0.660411,2,1,0,0.745584,HOLD_DO_NOT_SUBMIT,flex_archaludon_0010_minus1147_plus1192:HOLD_DO_NOT_SUBMIT; same archetype pair lacks portfolio diversity\\nflex_archaludon_0010_minus1147_plus1192;flex_archaludon_0001_minus1122_plus8,flex_archaludon_0010_minus1147_plus1192,flex_archaludon_0001_minus1122_plus8,archaludon,archaludon,1,0.875,0.673559,0.687897,0,1,0,0.743991,HOLD_DO_NOT_SUBMIT,flex_archaludon_0010_minus1147_plus1192:HOLD_DO_NOT_SUBMIT; same archetype pair lacks portfolio diversity\\nflex_archaludon_0010_minus1147_plus1192;flex_alakazam_dunsparce_0002_minus5_plus1266,flex_archaludon_0010_minus1147_plus1192,flex_alakazam_dunsparce_0002_minus5_plus1266,archaludon,alakazam_dunsparce,0,0.875,0.635409,0.654484,2,1,0,0.737286,HOLD_DO_NOT_SUBMIT,flex_archaludon_0010_minus1147_plus1192:HOLD_DO_NOT_SUBMIT\\nflex_archaludon_0010_minus1147_plus1192;flex_archaludon_0018_minus1182_plus1213,flex_archaludon_0010_minus1147_plus1192,flex_archaludon_0018_minus1182_plus1213,archaludon,archaludon,1,0.875,0.673559,0.703805,0,0,0,0.733991,HOLD_DO_NOT_SUBMIT,flex_archaludon_0010_minus1147_plus1192:HOLD_DO_NOT_SUBMIT; same archetype pair lacks portfolio diversity; no unique-edge complement\\nflex_archaludon_0010_minus1147_plus1192;flex_alakazam_dunsparce_0006_minus5_plus1156,flex_archaludon_0010_minus1147_plus1192,flex_alakazam_dunsparce_0006_minus5_plus1156,archaludon,alakazam_dunsparce,0,0.875,0.620151,0.646855,2,1,0,0.726606,HOLD_DO_NOT_SUBMIT,flex_archaludon_0010_minus1147_plus1192:HOLD_DO_NOT_SUBMIT\\nflex_archaludon_0018_minus1182_plus1213;flex_archaludon_0021_minus1182_plus1087,flex_archaludon_0018_minus1182_plus1213,flex_archaludon_0021_minus1182_plus1087,archaludon,archaludon,1,0.825,0.647263,0.690657,1,0,0,0.710584,HOLD_DO_NOT_SUBMIT,same archetype pair lacks portfolio diversity\\n", "promotion_counts": {"HOLD_DO_NOT_SUBMIT": 392, "REFERENCE_DERIVED_OPTION_C": 59, "NEED_MORE_DATA": 14}, "best_pair": {"A": "flex_archaludon_0018_minus1182_plus1213", "B": "flex_alakazam_dunsparce_0000_seed", "pair_held_score": 0.772424, "mean_best_score_rate": 0.875, "min_member_held_score": 0.685606, "final_decision": "NEED_MORE_DATA", "decision_mode": "CHALLENGER_BY_USER_APPROVAL"}}')
AGENT_PAYLOADS = json.loads('{"A": {"label": "A - Archaludon Metal Tempo Challenger", "emoji": "🛡️📈", "build_name": "flex_archaludon_0018_minus1182_plus1213", "archetype": "archaludon", "role": "Primary challenger: metal-tempo pressure into the current field.", "short": "Best eligible Archaludon-family pair member after coverage, holdout, and debias gates.", "strategy": "Use Cinderace tempo acceleration, Duraludon setup, Archaludon ex pressure, and Full Metal Lab durability to punish slower setup shells.", "risk": "Still provisional live-reference evidence; not a fully proven replacement until challenger feedback arrives.", "selector_comment": "A: 🛡️📈 Archaludon metal-tempo challenger — strongest eligible local/holdout pair member; provisional live gate.", "main_py": "\\"\\"\\"Archaludon ex + Cinderace — Rule-based agent (Public version)\\n\\nDeck Concept:\\n  Cinderace\'s Explosiveness places it face-down as Active during setup.\\n  Turn 1 Turbo Flare ({C}=50) accelerates up to 3 Basic Energy from deck\\n  to benched Duraludon. Evolving into Archaludon ex triggers Assemble Alloy,\\n  attaching up to 2 Basic Metal Energy from discard to Metal Pokemon.\\n  Metal Defender ({M}{M}{M}=220) is the main attack; no Weakness next turn.\\n  Duraludon can attack directly with Raging Hammer ({M}{M}{C}=80 + 10 per\\n  damage counter) without evolving. Relicanth\'s Memory Dive also unlocks\\n  Raging Hammer on Archaludon ex after evolution. Hero\'s Cape gives +100 HP\\n  (HP400). Full Metal Lab reduces attack damage to Metal Pokemon by 30.\\n\\nPokemon:\\n  Duraludon (169)      - Basic Metal HP130. Hammer In {M}=30.\\n                         Raging Hammer {M}{M}{C}=80+10*damage_counters.\\n  Archaludon ex (190)  - Stage 1 from Duraludon, HP300. Assemble Alloy: on evolve\\n                         from hand, attach up to 2 Metal Energy from discard.\\n                         Metal Defender {M}{M}{M}=220, no Weakness next turn.\\n  Cinderace (666)      - Stage 2 HP160. Explosiveness: place face-down as Active\\n                         in setup from opening hand. Turbo Flare {C}=50, attach\\n                         up to 3 Basic Energy from deck to benched Pokemon.\\n  Relicanth (57)       - Basic HP100. Memory Dive: evolved Pokemon can use attacks\\n                         from previous Evolutions. Archaludon ex -> Raging Hammer.\\n\\nTrainers:\\n  Poke Pad (1152), Ultra Ball (1121), Pokegear 3.0 (1122), Night Stretcher (1097),\\n  Jumbo Ice Cream (1147), Hero\'s Cape (1159), Boss\'s Orders (1182),\\n  Explorer\'s Guidance (1185), Lillie\'s Determination (1227), Full Metal Lab (1244) x4.\\n\\nEnergy: Basic Metal Energy (8) x11\\n\\nScore system:\\n  Setup/play/evolve/attach: 1000~28000 (high = do first)\\n  Attack: damage value (always last — attacking ends the turn)\\n  Negative = skip if above minCount\\n\\"\\"\\"\\n\\nimport os\\nimport random\\nimport sys\\n\\ntry:\\n    ROOT = __file__\\nexcept NameError:\\n    ROOT = None\\nCG_PATH = \\"/kaggle_simulations/agent\\"\\nfor p in ([os.path.dirname(os.path.abspath(ROOT))] if ROOT else []) + [CG_PATH]:\\n    if p and p not in sys.path and os.path.isdir(p):\\n        sys.path.insert(0, p)\\n\\nfrom cg.api import (\\n    AreaType,\\n    LogType,\\n    OptionType,\\n    SelectContext,\\n    all_card_data,\\n    to_observation_class,\\n)\\n\\ntry:\\n    from cg.api import all_attack\\n    ALL_ATTACKS = {a.attackId: a for a in all_attack()}\\nexcept Exception:\\n    ALL_ATTACKS = {}\\n\\n# ── Card IDs ──\\n\\nDURALUDON = 169\\nARCHALUDON_EX = 190\\nCINDERACE = 666\\nRELICANTH = 57\\nCRUSTLE_LINE = {344, 345, 532}\\nSTARMIE_LINE = {1030, 1031}\\nLUCARIO_LINE = {677, 678}\\nHOP_LINE = {288, 289, 299, 304, 307, 308, 309, 310, 878, 879}\\nHOP_SNORLAX = 304\\n\\nMETAL_ENERGY = 8\\n\\nPOKE_PAD = 1152\\nULTRA_BALL = 1121\\nPOKEGEAR = 1122\\nNIGHT_STRETCHER = 1097\\nJUMBO_ICE_CREAM = 1147\\nHERO_CAPE = 1159\\nBOSS = 1182\\nEXPLORER = 1185\\nLILLIE = 1227\\nFULL_METAL_LAB = 1244\\n\\nRAGING_HAMMER = 224\\nMETAL_DEFENDER = 253\\n\\n_ATTACK_BASE_DMG = {METAL_DEFENDER: 220, 965: 50, 223: 30, 61: 30}\\n\\n_SETUP_ACTIVE_PRIORITY = {\\n    CINDERACE: (100000, \\"Active: Cinderace Explosiveness\\"),\\n    DURALUDON: (20000, \\"Active fallback: Duraludon\\"),\\n    RELICANTH: (5000, \\"Active fallback: Relicanth\\"),\\n}\\n\\nALWAYS_SAFE_DISCARD = {METAL_ENERGY, CINDERACE}\\n\\nCARD_DB = {c.cardId: c for c in all_card_data()}\\n\\nMEGA_BRAVE = 983\\nPREMIUM_POWER_PRO = 1141\\nHARIYAMA_LINE = {673, 674}\\n\\n# Track opponent\'s last-turn attack via logs\\n_opp_last_attack_id = None\\n_cur_turn_logs = []\\n\\n\\ndef _update_opp_attack_tracking(obs):\\n    global _opp_last_attack_id, _cur_turn_logs\\n    yi = obs.current.yourIndex\\n    for entry in obs.logs:\\n        if entry.type == LogType.TURN_END:\\n            for prev in _cur_turn_logs:\\n                if prev.type == LogType.ATTACK and getattr(prev, \'playerIndex\', yi) != yi:\\n                    _opp_last_attack_id = prev.attackId\\n            _cur_turn_logs.clear()\\n        else:\\n            _cur_turn_logs.append(entry)\\n\\n\\n# ── Board helpers ──\\n\\ndef read_deck_csv():\\n    fp = \\"deck.csv\\"\\n    if not os.path.exists(fp):\\n        fp = \\"/kaggle_simulations/agent/deck.csv\\"\\n    with open(fp) as f:\\n        return [int(line) for line in f.read().strip().split(\\"\\\\n\\")]\\n\\n\\ndef get_card(obs, area, index, player_index):\\n    if area is None or index is None:\\n        return None\\n    ps = obs.current.players[player_index]\\n    if area == AreaType.DECK and obs.select and obs.select.deck is not None:\\n        return obs.select.deck[index] if index < len(obs.select.deck) else None\\n    if area == AreaType.HAND and ps.hand is not None:\\n        return ps.hand[index] if index < len(ps.hand) else None\\n    if area == AreaType.DISCARD:\\n        return ps.discard[index] if index < len(ps.discard) else None\\n    if area == AreaType.ACTIVE:\\n        return ps.active[index] if index < len(ps.active) else None\\n    if area == AreaType.BENCH:\\n        return ps.bench[index] if index < len(ps.bench) else None\\n    if area == AreaType.PRIZE:\\n        return ps.prize[index] if index < len(ps.prize) else None\\n    if area == AreaType.STADIUM:\\n        return obs.current.stadium[index] if index < len(obs.current.stadium) else None\\n    if area == AreaType.LOOKING and obs.current.looking is not None:\\n        return obs.current.looking[index] if index < len(obs.current.looking) else None\\n    return None\\n\\n\\ndef option_card(obs, opt):\\n    yi = obs.current.yourIndex\\n    pi = opt.playerIndex if opt.playerIndex is not None else yi\\n    if opt.type == OptionType.PLAY:\\n        return get_card(obs, AreaType.HAND, opt.index, pi)\\n    return get_card(obs, opt.area, opt.index, pi)\\n\\n\\ndef option_target(obs, opt):\\n    if opt.inPlayArea is None or opt.inPlayIndex is None:\\n        return None\\n    return get_card(obs, opt.inPlayArea, opt.inPlayIndex, obs.current.yourIndex)\\n\\n\\ndef my_state(obs):\\n    return obs.current.players[obs.current.yourIndex]\\n\\n\\ndef opp_state(obs):\\n    return obs.current.players[1 - obs.current.yourIndex]\\n\\n\\ndef active_pokemon(obs):\\n    ps = my_state(obs)\\n    return ps.active[0] if ps.active else None\\n\\n\\ndef opp_active_pokemon(obs):\\n    ps = opp_state(obs)\\n    return ps.active[0] if ps.active else None\\n\\n\\ndef opp_bench_pokemon(obs):\\n    return [p for p in opp_state(obs).bench if p]\\n\\n\\ndef all_my_pokemon(obs):\\n    ps = my_state(obs)\\n    return [p for p in (ps.active + ps.bench) if p]\\n\\n\\ndef hand_ids(obs):\\n    hand = my_state(obs).hand\\n    return [c.id for c in hand if c] if hand else []\\n\\n\\ndef discard_ids(obs):\\n    return [c.id for c in (my_state(obs).discard or []) if c]\\n\\n\\ndef metal_in_discard(obs):\\n    return sum(1 for c in (my_state(obs).discard or []) if c and c.id == METAL_ENERGY)\\n\\n\\ndef energy_count(pokemon):\\n    if pokemon is None:\\n        return 0\\n    if getattr(pokemon, \\"energyCards\\", None) is not None:\\n        return len(pokemon.energyCards)\\n    return len(getattr(pokemon, \\"energies\\", []) or [])\\n\\n\\ndef retreat_cost(pokemon):\\n    data = CARD_DB.get(pokemon.id) if pokemon else None\\n    return getattr(data, \\"retreatCost\\", 0) if data else 0\\n\\n\\ndef damage_on(pokemon):\\n    if pokemon is None:\\n        return 0\\n    return max(0, getattr(pokemon, \\"maxHp\\", pokemon.hp) - pokemon.hp)\\n\\n\\ndef has_tool(pokemon):\\n    return bool(getattr(pokemon, \\"tools\\", []) or [])\\n\\n\\ndef count_in_play(obs, card_id):\\n    return sum(1 for p in all_my_pokemon(obs) if p.id == card_id)\\n\\n\\ndef has_in_play(obs, card_id):\\n    return any(p.id == card_id for p in all_my_pokemon(obs))\\n\\n\\ndef need_duraludon(obs):\\n    return sum(1 for p in all_my_pokemon(obs) if p.id in {DURALUDON, ARCHALUDON_EX}) < 2\\n\\n\\ndef need_archaludon(obs):\\n    has_dura, ex_count = False, 0\\n    for p in all_my_pokemon(obs):\\n        if p.id == DURALUDON:\\n            has_dura = True\\n        elif p.id == ARCHALUDON_EX:\\n            ex_count += 1\\n    return has_dura and ex_count < 2\\n\\n\\ndef safe_discard_count(obs):\\n    ids = hand_ids(obs)\\n    mt = metal_in_discard(obs)\\n    safe = 0\\n    for cid in ids:\\n        if cid == METAL_ENERGY and mt + safe < 2:\\n            safe += 1\\n        elif cid == CINDERACE:\\n            safe += 1\\n    draw_in_hand = sum(1 for c in ids if c in (LILLIE, EXPLORER))\\n    if draw_in_hand >= 2:\\n        safe += draw_in_hand - 1\\n    return safe\\n\\n\\ndef prize_value(pokemon):\\n    data = CARD_DB.get(pokemon.id) if pokemon else None\\n    if data and getattr(data, \\"megaEx\\", False):\\n        return 3\\n    if data and getattr(data, \\"ex\\", False):\\n        return 2\\n    return 1\\n\\n\\ndef best_attack_damage(obs, attack_id):\\n    if attack_id == RAGING_HAMMER:\\n        return 80 + damage_on(active_pokemon(obs)) // 10 * 10\\n    return _ATTACK_BASE_DMG.get(attack_id, 0)\\n\\n\\ndef is_metal_weak(pokemon):\\n    if pokemon is None:\\n        return False\\n    data = CARD_DB.get(pokemon.id)\\n    w = getattr(data, \\"weakness\\", None) if data else None\\n    if w is None:\\n        return False\\n    return getattr(w, \\"value\\", w) == METAL_ENERGY\\n\\n\\ndef effective_damage(base_damage, target):\\n    return base_damage * 2 if is_metal_weak(target) else base_damage\\n\\n\\ndef _first_option_index(obs, card_id):\\n    for o in obs.select.option:\\n        oc = option_card(obs, o)\\n        if oc and oc.id == card_id:\\n            return getattr(o, \'index\', None)\\n    return None\\n\\n\\n# ── Attack routes ──\\n\\ndef direct_attack_energy_route(obs, pokemon):\\n    e = energy_count(pokemon)\\n    if e >= 3:\\n        return True, False\\n    if e == 2 and not obs.current.energyAttached and METAL_ENERGY in hand_ids(obs):\\n        return True, True\\n    return False, False\\n\\n\\ndef can_evolve_to_archaludon_now(pokemon, obs):\\n    if pokemon is None or pokemon.id != DURALUDON:\\n        return False\\n    if ARCHALUDON_EX not in hand_ids(obs):\\n        return False\\n    return not getattr(pokemon, \\"appearThisTurn\\", True)\\n\\n\\ndef alloy_attack_energy_route(obs, pokemon):\\n    if not can_evolve_to_archaludon_now(pokemon, obs):\\n        return False, False\\n    current = energy_count(pokemon)\\n    alloy = min(2, metal_in_discard(obs))\\n    total = current + alloy\\n    if total >= 3:\\n        return True, False\\n    if total == 2 and not obs.current.energyAttached and METAL_ENERGY in hand_ids(obs):\\n        return True, True\\n    return False, False\\n\\n\\ndef attack_energy_route(obs, pokemon):\\n    if pokemon is None:\\n        return False, False\\n    if pokemon.id == ARCHALUDON_EX:\\n        return direct_attack_energy_route(obs, pokemon)\\n    if pokemon.id == DURALUDON:\\n        ok, uses_attach = direct_attack_energy_route(obs, pokemon)\\n        if ok:\\n            return True, uses_attach\\n        return alloy_attack_energy_route(obs, pokemon)\\n    return False, False\\n\\n\\ndef archaludon_ex_attack_route(obs):\\n    active = active_pokemon(obs)\\n    if active and active.id in {ARCHALUDON_EX, DURALUDON}:\\n        ok, uses_attach = attack_energy_route(obs, active)\\n        if ok:\\n            return {\\"attacker\\": active, \\"uses_attach\\": uses_attach, \\"needs_retreat\\": False}\\n\\n    if active is None or obs.current.retreated or energy_count(active) < retreat_cost(active):\\n        return None\\n    ps = my_state(obs)\\n    for pokemon in [p for p in ps.bench if p]:\\n        if pokemon.id not in {ARCHALUDON_EX, DURALUDON}:\\n            continue\\n        ok, uses_attach = attack_energy_route(obs, pokemon)\\n        if ok:\\n            return {\\"attacker\\": pokemon, \\"uses_attach\\": uses_attach, \\"needs_retreat\\": True}\\n    return None\\n\\n\\ndef planned_archaludon_attacks(obs):\\n    route = archaludon_ex_attack_route(obs)\\n    if route is None:\\n        return []\\n    attacker = route[\\"attacker\\"]\\n    attacks = []\\n    if attacker.id == ARCHALUDON_EX:\\n        attacks.append({\\"damage\\": 220})\\n        if has_in_play(obs, RELICANTH):\\n            attacks.append({\\"damage\\": 80 + damage_on(attacker) // 10 * 10})\\n    if attacker.id == DURALUDON:\\n        attacks.append({\\"damage\\": 80 + damage_on(attacker) // 10 * 10})\\n        if can_evolve_to_archaludon_now(attacker, obs):\\n            attacks.append({\\"damage\\": 220})\\n    return attacks\\n\\n\\n# ── Matchup detection & opponent max damage ──\\n\\nALAKAZAM_LINE = {741, 742, 743}\\n_ALA_BOARD_GAIN = {66: 3, 742: 2, 305: 2, 65: 2, 741: 1}  # Dudunsparce, Kadabra, Dunsparce×2, Abra\\n\\n\\ndef _estimate_alakazam_from_pokes(opp, pokes):\\n    \\"\\"\\"(floor, ceiling, ceiling_with_boss) damage from visible Alakazam line.\\"\\"\\"\\n    ids = [p.id for p in pokes if p]\\n    if not (ALAKAZAM_LINE & set(ids)):\\n        return 0, 0, 0\\n    base = opp.handCount + 1\\n    gain = sum(_ALA_BOARD_GAIN.get(i, 0) for i in ids)\\n    enriching_seen = (\\n        any(c and c.id == 13 for c in (opp.discard or []))\\n        or any(c and c.id == 13 for p in pokes if p for c in (getattr(p, \\"energyCards\\", None) or []))\\n    )\\n    if not enriching_seen:\\n        gain += 3\\n    if any(i == 140 for i in ids):\\n        gain += 3\\n    return base * 20, (base + gain + 2) * 20, (base + gain - 1) * 20\\n\\n\\ndef _estimate_alakazam(obs):\\n    \\"\\"\\"(floor, ceiling, ceiling_with_boss) damage from Powerful Hand.\\"\\"\\"\\n    opp = opp_state(obs)\\n    pokes = ([opp.active[0]] if opp.active else []) + list(opp.bench or [])\\n    return _estimate_alakazam_from_pokes(opp, pokes)\\n\\n\\ndef detect_matchup(obs):\\n    opp = opp_state(obs)\\n    ids = {p.id for p in (opp.active + opp.bench) if p}\\n    if ids & CRUSTLE_LINE:\\n        return \\"crustle\\"\\n    if ids & HOP_LINE:\\n        return \\"hop\\"\\n    if ids & STARMIE_LINE:\\n        return \\"starmie\\"\\n    if ids & LUCARIO_LINE:\\n        return \\"lucario\\"\\n    if ids & ALAKAZAM_LINE:\\n        return \\"alakazam\\"\\n    return \\"generic\\"\\n\\n\\ndef opp_max_damage(obs):\\n    matchup = detect_matchup(obs)\\n    if matchup == \\"alakazam\\":\\n        _, ceiling, _ = _estimate_alakazam(obs)\\n        return ceiling\\n    if matchup == \\"crustle\\":\\n        return 120\\n    if matchup == \\"hop\\":\\n        return 220\\n    if matchup == \\"lucario\\":\\n        return 270  # Mega Brave base. PPP adds +30 each but unpredictable\\n    if matchup == \\"starmie\\":\\n        return 210\\n    return 220\\n\\n\\n# ── Overrides ──\\n\\ndef apply_overrides(obs, opt, score, reason):\\n    # Hard rule: don\'t Explorer with low deck\\n    if opt.type == OptionType.PLAY:\\n        card = option_card(obs, opt)\\n        cid = card.id if card else None\\n        if my_state(obs).deckCount <= 10 and cid == EXPLORER:\\n            return -5000, \\"hard: don\'t Explorer with low deck\\"\\n\\n    if detect_matchup(obs) != \\"crustle\\":\\n        return score, reason\\n\\n    # Crustle overrides\\n    card = option_card(obs, opt)\\n    cid = card.id if card else getattr(opt, \'cardId\', None)\\n    ctx = obs.select.context\\n\\n    if opt.type == OptionType.EVOLVE and cid == ARCHALUDON_EX:\\n        return -10000, \\"Crustle: don\'t evolve to ex\\"\\n\\n    if opt.type == OptionType.ATTACK:\\n        aid = getattr(opt, \'attackId\', None)\\n        active = active_pokemon(obs)\\n        opp_act = opp_active_pokemon(obs)\\n        opp_has_spiky = bool(opp_act and any(\\n            getattr(c, \'id\', None) == 14\\n            for c in (getattr(opp_act, \'energyCards\', None) or [])))\\n        if (active and active.id == DURALUDON and active.hp == 130\\n                and opp_act and opp_act.id == 345 and energy_count(opp_act) >= 2\\n                and opp_has_spiky):\\n            return -3000, \\"Crustle: full HP Duraludon waits out Spiky\\"\\n        if aid == METAL_DEFENDER:\\n            return -5000, \\"Crustle: Metal Defender does 0\\"\\n        if aid == RAGING_HAMMER:\\n            rh_dmg = 80 + damage_on(active_pokemon(obs)) // 10 * 10\\n            return max(score, 200), \\"Crustle: Raging Hammer\\"\\n\\n    if opt.type == OptionType.PLAY:\\n        if cid == RELICANTH:\\n            return -5000, \\"Crustle: skip Relicanth\\"\\n        dc = my_state(obs).deckCount\\n        if dc <= 10 and cid in (EXPLORER, LILLIE):\\n            if cid == LILLIE and dc <= 3 and my_state(obs).handCount >= dc + 6:\\n                return 15000, \\"Crustle: Lillie to refill deck\\"\\n            return -5000, \\"Crustle: don\'t draw with low deck\\"\\n        if cid == LILLIE:\\n            has_metal = any(c and c.id == METAL_ENERGY for c in (my_state(obs).hand or []) if c)\\n            if not has_metal:\\n                return score, \\"Crustle: Lillie OK (no energy in hand)\\"\\n\\n    if opt.type == OptionType.ATTACH:\\n        target = option_target(obs, opt)\\n        tid = target.id if target else None\\n        if getattr(opt, \'inPlayArea\', None) == AreaType.BENCH and tid == DURALUDON:\\n            return score + 10000, \\"Crustle: bench Duraludon energy priority\\"\\n        if getattr(opt, \'inPlayArea\', None) == AreaType.ACTIVE:\\n            active = active_pokemon(obs)\\n            if active and energy_count(active) >= 2:\\n                return score + 3000, \\"Crustle: Active 3rd energy\\"\\n\\n    if ctx == SelectContext.TO_HAND and opt.type == OptionType.CARD and cid == ARCHALUDON_EX:\\n        return -3000, \\"Crustle: skip Archaludon ex\\"\\n\\n    if ctx in {SelectContext.DISCARD, SelectContext.DISCARD_CARD_OR_ATTACHED_CARD}:\\n        if cid == ARCHALUDON_EX and score < 0:\\n            return 9000, \\"Crustle: discard Archaludon ex\\"\\n\\n    return score, reason\\n\\n\\n# ── Scoring ──\\n\\ndef score_setup(obs, opt):\\n    card = option_card(obs, opt)\\n    cid = card.id if card else None\\n    ctx = obs.select.context\\n\\n    if ctx == SelectContext.MULLIGAN:\\n        return (10000, \\"no mulligan\\") if opt.type == OptionType.NO else (0, \\"mulligan\\")\\n    if ctx == SelectContext.IS_FIRST:\\n        return (10000, \\"choose second\\") if opt.type == OptionType.NO else (0, \\"go first\\")\\n    if ctx == SelectContext.SETUP_ACTIVE_POKEMON:\\n        return _SETUP_ACTIVE_PRIORITY.get(cid, (0, \\"unknown Active\\"))\\n    if ctx == SelectContext.SETUP_BENCH_POKEMON:\\n        return -10000, \\"never bench during setup\\"\\n    return 0, \\"non-setup\\"\\n\\n\\n# HP threshold per matchup: skip Ice Cream if HP > this value\\n_ICE_CREAM_HP_THRESHOLD = {\\n    \\"lucario\\": 270,\\n    \\"starmie\\": 210,\\n    \\"crustle\\": 120,\\n    \\"hop\\": 220,\\n    \\"generic\\": 230,\\n}\\n\\n\\ndef should_skip_ice_cream(obs, active):\\n    \\"\\"\\"Decide whether to skip Jumbo Ice Cream. Returns (skip: bool, reason: str).\\"\\"\\"\\n    # 1. Active must be Archaludon ex\\n    if active.id != ARCHALUDON_EX:\\n        return True, \\"skip Ice Cream: not Archaludon ex\\"\\n    # 2. Raging Hammer KO guard: don\'t heal if it loses a KO (but 220 Metal Defender still KOs → heal OK)\\n    opp_act = opp_active_pokemon(obs)\\n    if opp_act and has_in_play(obs, RELICANTH):\\n        md_kills = effective_damage(220, opp_act) >= opp_act.hp\\n        if not md_kills:\\n            rh_dmg = 80 + damage_on(active) // 10 * 10\\n            rh_after = 80 + max(0, damage_on(active) - 80) // 10 * 10\\n            if effective_damage(rh_dmg, opp_act) >= opp_act.hp and effective_damage(rh_after, opp_act) < opp_act.hp:\\n                return True, \\"skip Ice Cream: healing loses Raging Hammer KO\\"\\n    # 3. Alakazam: all-or-nothing Ice Cream decision\\n    matchup = detect_matchup(obs)\\n    if matchup == \\"alakazam\\":\\n        floor, ceiling, _ = _estimate_alakazam(obs)\\n        opp_a = opp_active_pokemon(obs)\\n        attacks = planned_archaludon_attacks(obs)\\n        if opp_a and attacks and any(effective_damage(a[\\"damage\\"], opp_a) >= opp_a.hp for a in attacks):\\n            _, ceiling, _ = _estimate_alakazam_from_pokes(opp_state(obs), opp_bench_pokemon(obs))\\n        ice_count = sum(1 for c in (my_state(obs).hand or []) if c and c.id == JUMBO_ICE_CREAM)\\n        max_hp = getattr(active, \\"maxHp\\", active.hp)\\n        hp_after_all = min(max_hp, active.hp + ice_count * 80)\\n        if hp_after_all <= active.hp:\\n            return True, \\"skip Ice Cream: no effective healing\\"\\n        if hp_after_all < floor:\\n            return True, f\\"skip Ice Cream: even {ice_count}x heal ({hp_after_all}) < floor {floor}\\"\\n        if hp_after_all >= ceiling:\\n            return False, f\\"use Ice Cream: {ice_count}x heal ({hp_after_all}) >= ceil {ceiling}\\"\\n        return False, f\\"use Ice Cream: {ice_count}x heal ({hp_after_all}) between floor={floor} ceil={ceiling}\\"\\n    # 4. HP above matchup threshold\\n    threshold = _ICE_CREAM_HP_THRESHOLD.get(matchup, 220)\\n    if active.hp > threshold:\\n        return True, f\\"skip Ice Cream: HP {active.hp} > {threshold} ({matchup})\\"\\n    # 5. Use it\\n    return False, \\"\\"\\n\\n\\nITEMS = {POKE_PAD, ULTRA_BALL, POKEGEAR, NIGHT_STRETCHER, JUMBO_ICE_CREAM, HERO_CAPE}\\n\\n\\ndef score_play(obs, opt):\\n    card = option_card(obs, opt)\\n    cid = card.id if card else None\\n    ids = hand_ids(obs)\\n\\n    # ── Pokemon: bench if available ──\\n    if cid in {DURALUDON, RELICANTH}:\\n        return 18000, \\"play Pokemon\\"\\n\\n    # ── Stadium ──\\n    if cid == FULL_METAL_LAB:\\n        active = active_pokemon(obs)\\n        if active and active.id not in {DURALUDON, ARCHALUDON_EX}:\\n            return -200, \\"skip FML: Active not Metal\\"\\n        return 20000, \\"play Full Metal Lab\\"\\n\\n    # ── Items: default 20000, only negative exceptions ──\\n    if cid in ITEMS:\\n        if cid == HERO_CAPE:\\n            if not any(p.id in {ARCHALUDON_EX, DURALUDON} and not has_tool(p) for p in all_my_pokemon(obs)):\\n                return -500, \\"save Hero\'s Cape: no target\\"\\n        if cid == JUMBO_ICE_CREAM:\\n            active = active_pokemon(obs)\\n            if active:\\n                skip, reason = should_skip_ice_cream(obs, active)\\n                if skip:\\n                    return -500, reason\\n        if cid == NIGHT_STRETCHER:\\n            disc = discard_ids(obs)\\n            has_urgent = (\\n                (DURALUDON in disc and DURALUDON not in ids and count_in_play(obs, DURALUDON) + count_in_play(obs, ARCHALUDON_EX) <= 1)\\n                or (ARCHALUDON_EX in disc and ARCHALUDON_EX not in ids and has_in_play(obs, DURALUDON))\\n                or (METAL_ENERGY in disc and not obs.current.energyAttached\\n                    and sum(1 for c in (my_state(obs).hand or []) if c and c.id == METAL_ENERGY) == 0\\n                    and any(p and p.id in (DURALUDON, ARCHALUDON_EX) and energy_count(p) == 2 for p in all_my_pokemon(obs)))\\n            )\\n            if not has_urgent:\\n                return -500, \\"save Night Stretcher\\"\\n        if cid == ULTRA_BALL:\\n            bench_empty = len([p for p in my_state(obs).bench if p]) == 0\\n            if bench_empty:\\n                return 300, \\"Ultra Ball: bench empty (donk risk)\\"\\n            metal_in_hand = sum(1 for c in (my_state(obs).hand or []) if c and c.id == METAL_ENERGY)\\n            metal_in_trash = metal_in_discard(obs)\\n            if metal_in_trash == 0 and metal_in_hand >= 1:\\n                return 20000, \\"Ultra Ball: fuel Alloy\\"\\n            if safe_discard_count(obs) >= 2 and (need_archaludon(obs) or need_duraludon(obs)):\\n                return 20000, \\"Ultra Ball: search line\\"\\n            return -1000, \\"skip Ultra Ball\\"\\n        return 20000, \\"play item\\"\\n\\n    if cid == EXPLORER:\\n        if obs.current.supporterPlayed:\\n            return -1000, \\"Supporter already used\\"\\n        return 16000, \\"play Explorer\\"\\n\\n    if cid == LILLIE:\\n        if obs.current.supporterPlayed:\\n            return -1000, \\"Supporter already used\\"\\n        if BOSS in ids and planned_archaludon_attacks(obs):\\n            return -500, \\"save Lillie: Boss in hand with attacker ready\\"\\n        return 5000, \\"play Lillie\\"\\n\\n    if cid == BOSS:\\n        if obs.current.supporterPlayed:\\n            return -1000, \\"Supporter already used\\"\\n        # vs Hop: Boss Snorlax to remove Extra Helpings (+30) ASAP\\n        if detect_matchup(obs) == \\"hop\\":\\n            active = active_pokemon(obs)\\n            opp_has_snorlax = any(p.id == HOP_SNORLAX for p in opp_bench_pokemon(obs))\\n            if opp_has_snorlax and active:\\n                # Case 1: Cinderace active + bench has Duraludon → Turbo Flare Snorlax\\n                if active.id == CINDERACE:\\n                    has_dura_bench = any(p.id in {DURALUDON, ARCHALUDON_EX}\\n                                        for p in my_state(obs).bench if p)\\n                    if has_dura_bench:\\n                        return 16500, \\"Boss: pull Snorlax (Cinderace Turbo Flare)\\"\\n                # Case 2: Archaludon active, HP > 220, can attack → Boss Snorlax\\n                if active.id == ARCHALUDON_EX and active.hp > 220:\\n                    ok, _ = attack_energy_route(obs, active)\\n                    if ok:\\n                        return 16500, \\"Boss: pull Snorlax (Arch can tank Revenge 220)\\"\\n        if _opp_last_attack_id == MEGA_BRAVE:\\n            return -500, \\"save Boss: Mega Brave stuck\\"\\n        attacks = planned_archaludon_attacks(obs)\\n        if not attacks:\\n            return -500, \\"save Boss: no attacker\\"\\n        opp_act = opp_active_pokemon(obs)\\n        can_ko_active = opp_act and any(\\n            effective_damage(atk[\\"damage\\"], opp_act) >= opp_act.hp for atk in attacks)\\n        remaining = len(my_state(obs).prize)\\n        if can_ko_active:\\n            if prize_value(opp_act) >= remaining:\\n                return -500, \\"save Boss: Active KO wins\\"\\n            for target in opp_bench_pokemon(obs):\\n                for atk in attacks:\\n                    if effective_damage(atk[\\"damage\\"], target) >= target.hp:\\n                        if prize_value(target) >= remaining:\\n                            return 20000, \\"LETHAL Boss\\"\\n                        break\\n            return -500, \\"save Boss: can KO Active\\"\\n        best_score = -500\\n        best_reason = \\"save Boss\\"\\n        for target in opp_bench_pokemon(obs):\\n            for atk in attacks:\\n                if effective_damage(atk[\\"damage\\"], target) >= target.hp:\\n                    pv = prize_value(target)\\n                    if pv >= remaining:\\n                        return 20000, \\"LETHAL Boss\\"\\n                    s = 4000 + pv * 200 + energy_count(target) * 100\\n                    if s > best_score:\\n                        best_score = s\\n                        best_reason = \\"Boss: pull bench target\\"\\n                    break\\n        if best_score <= 0:\\n            metal_total = sum(1 for c in (my_state(obs).hand or []) if c and c.id == METAL_ENERGY)\\n            metal_total += sum(energy_count(p) for p in all_my_pokemon(obs) if p)\\n            has_cind = has_in_play(obs, CINDERACE)\\n            draw_in_hand = any(c and c.id in (EXPLORER, LILLIE) for c in (my_state(obs).hand or []) if c)\\n            if metal_total <= 2 and not has_cind and not draw_in_hand:\\n                best_stall = -500\\n                stall_reason = \\"save Boss\\"\\n                for target in opp_bench_pokemon(obs):\\n                    te = energy_count(target)\\n                    cd = CARD_DB.get(target.id)\\n                    rc = cd.retreatCost if cd else 0\\n                    min_atk = 99\\n                    if cd and cd.attacks:\\n                        for aid in cd.attacks:\\n                            atk = ALL_ATTACKS.get(aid)\\n                            if atk:\\n                                min_atk = min(min_atk, len(atk.energies))\\n                    if min_atk == 99:\\n                        min_atk = 1\\n                    ss = 4000 + rc * 1000 + min_atk * 500 - te * 800\\n                    if ss > best_stall:\\n                        best_stall = ss\\n                        stall_reason = \\"Boss stall\\"\\n                return best_stall, stall_reason\\n        return best_score, best_reason\\n\\n    return 1000, \\"generic play\\"\\n\\n\\ndef score_evolve(obs, opt):\\n    card = option_card(obs, opt)\\n    target = option_target(obs, opt)\\n    cid = card.id if card else None\\n    tid = target.id if target else None\\n    if cid == ARCHALUDON_EX and tid == DURALUDON:\\n        target_is_active = opt.inPlayArea == AreaType.ACTIVE\\n        mc = metal_in_discard(obs)\\n        if target_is_active:\\n            if energy_count(target) >= 3 and not has_in_play(obs, ARCHALUDON_EX):\\n                return 17000, \\"evolve Active 3-energy Duraludon\\"\\n            if mc >= 2:\\n                return 28000 + mc * 2000, \\"evolve Active Duraludon\\"\\n            if mc == 1:\\n                return 8000, \\"delay Active evolve: 1 Metal\\"\\n            return -500, \\"hold: no Metal in discard\\"\\n        if mc >= 2:\\n            return 14000 + mc * 1000, \\"evolve Bench Duraludon\\"\\n        return -1000, \\"hold: evolve Active first\\"\\n    return 10000, \\"generic evolution\\"\\n\\n\\ndef attach_target_score(obs, target, area):\\n    if target is None:\\n        return 0\\n    cid = target.id\\n    e = energy_count(target)\\n\\n    if e >= 3:\\n        return -5000\\n    if cid == CINDERACE and e >= 1:\\n        return -3000\\n\\n    score = 0\\n    if cid == CINDERACE:\\n        score = 3000\\n        if e == 0:\\n            score += 7000 + (12000 if area == AreaType.ACTIVE else 5000)\\n    elif cid in {DURALUDON, ARCHALUDON_EX}:\\n        score = 6000 if cid == ARCHALUDON_EX else 5500\\n        score += {2: 12000, 1: 7000, 0: 4000}.get(e, -1000)\\n        score += 1000 if area == AreaType.ACTIVE else 500\\n    else:\\n        score = 1000 + (1000 if e == 0 else 0)\\n\\n    # HP-based adjustment\\n    if target.hp > 0:\\n        max_hp = getattr(target, \\"maxHp\\", target.hp)\\n        ratio = target.hp / max_hp if max_hp > 0 else 1\\n        if ratio <= 0.25:\\n            score -= 1500\\n        elif ratio <= 0.50:\\n            score -= 500\\n        else:\\n            score += min(1000, target.hp // 40 * 100)\\n    return score\\n\\n\\ndef score_attach(obs, opt):\\n    card = option_card(obs, opt)\\n    target = option_target(obs, opt)\\n    cid = card.id if card else None\\n    tid = target.id if target else None\\n\\n    if cid == HERO_CAPE:\\n        if tid == ARCHALUDON_EX and target and not has_tool(target):\\n            return 11000, \\"Hero\'s Cape on Archaludon ex\\"\\n        if tid == DURALUDON and target and not has_tool(target) and energy_count(target) >= 1:\\n            return 8000, \\"Hero\'s Cape on Duraludon\\"\\n        return -1000, \\"save Hero\'s Cape\\"\\n\\n    if cid != METAL_ENERGY:\\n        return -500, \\"skip non-Metal\\"\\n    if obs.current.energyAttached:\\n        return -1000, \\"already attached\\"\\n\\n    return attach_target_score(obs, target, opt.inPlayArea), \\"attach Metal\\"\\n\\n\\ndef score_retreat(obs, opt):\\n    active = active_pokemon(obs)\\n    if active and active.id == ARCHALUDON_EX and has_tool(active) and active.hp > 200:\\n        return -5000, \\"don\'t retreat HP400 tank\\"\\n    route = archaludon_ex_attack_route(obs)\\n    if route and route[\\"needs_retreat\\"]:\\n        return 13000, \\"retreat to attack-ready ex\\"\\n    return -100, \\"avoid retreat\\"\\n\\n\\n_MAIN_DISPATCH = {\\n    OptionType.PLAY: score_play, OptionType.EVOLVE: score_evolve,\\n    OptionType.ATTACH: score_attach, OptionType.RETREAT: score_retreat,\\n}\\n\\n\\ndef score_option(obs, opt):\\n    ctx = obs.select.context\\n\\n    if ctx in {SelectContext.IS_FIRST, SelectContext.MULLIGAN,\\n               SelectContext.SETUP_ACTIVE_POKEMON, SelectContext.SETUP_BENCH_POKEMON}:\\n        return score_setup(obs, opt)\\n\\n    if opt.type in {OptionType.YES, OptionType.NO}:\\n        if ctx == SelectContext.IS_FIRST:\\n            return score_setup(obs, opt)\\n        if ctx == SelectContext.ACTIVATE:\\n            return (100000, \\"Explosiveness\\") if opt.type == OptionType.YES else (-100000, \\"never decline\\")\\n        return (1, \\"yes\\") if opt.type == OptionType.YES else (0, \\"no\\")\\n\\n    if opt.type == OptionType.NUMBER:\\n        return (opt.number or 0), \\"number\\"\\n\\n    if ctx == SelectContext.MAIN:\\n        fn = _MAIN_DISPATCH.get(opt.type)\\n        if fn:\\n            score, reason = fn(obs, opt)\\n        elif opt.type == OptionType.ABILITY:\\n            score, reason = 1, \\"ability\\"\\n        elif opt.type == OptionType.ATTACK:\\n            score, reason = best_attack_damage(obs, opt.attackId), \\"attack\\"\\n        elif opt.type == OptionType.END:\\n            score, reason = 0, \\"end turn\\"\\n        else:\\n            score, reason = 500, \\"generic MAIN\\"\\n    elif ctx == SelectContext.TO_HAND:\\n        score, reason = score_to_hand(obs, opt)\\n    elif ctx in {SelectContext.DISCARD, SelectContext.DISCARD_CARD_OR_ATTACHED_CARD}:\\n        score, reason = score_discard(obs, opt)\\n    elif ctx in {SelectContext.ATTACH_TO, SelectContext.TO_FIELD, SelectContext.TO_BENCH,\\n                 SelectContext.ATTACH_FROM, SelectContext.SWITCH, SelectContext.TO_ACTIVE,\\n                 SelectContext.HEAL, SelectContext.DAMAGE}:\\n        score, reason = score_target(obs, opt)\\n    elif ctx == SelectContext.ATTACK:\\n        score, reason = best_attack_damage(obs, opt.attackId), \\"attack\\"\\n    elif opt.type == OptionType.CARD:\\n        score, reason = score_to_hand(obs, opt)\\n    elif opt.type == OptionType.ENERGY:\\n        score, reason = 1000, \\"energy\\"\\n    elif opt.type == OptionType.END:\\n        score, reason = 0, \\"end\\"\\n    else:\\n        score, reason = 100, \\"fallback\\"\\n\\n    return apply_overrides(obs, opt, score, reason)\\n\\n\\ndef score_to_hand(obs, opt):\\n    card = option_card(obs, opt)\\n    cid = card.id if card else opt.cardId\\n    ids = hand_ids(obs)\\n    effect = getattr(obs.select, \\"effect\\", None)\\n    effect_id = effect.id if effect else None\\n\\n    if effect_id == EXPLORER:\\n        has_ready = any(p and p.id in (DURALUDON, ARCHALUDON_EX) and energy_count(p) >= 3\\n                        for p in all_my_pokemon(obs))\\n        metal_in_hand = sum(1 for c in (my_state(obs).hand or []) if c and c.id == METAL_ENERGY)\\n\\n        if cid == HERO_CAPE:\\n            has_target = any(p.id == ARCHALUDON_EX and not has_tool(p) for p in all_my_pokemon(obs))\\n            return (27000 if has_target else 22000), \\"Explorer: Hero\'s Cape\\"\\n        if cid == METAL_ENERGY:\\n            if has_ready or metal_in_hand > 0:\\n                return 0, \\"Explorer: skip energy\\"\\n            if getattr(opt, \'index\', 0) == _first_option_index(obs, METAL_ENERGY):\\n                return 25000, \\"Explorer: take 1st energy\\"\\n            return 0, \\"Explorer: skip 2nd energy\\"\\n        if cid == ARCHALUDON_EX and need_archaludon(obs):\\n            return 20000, \\"Explorer: take Archaludon ex\\"\\n        if cid == DURALUDON and need_duraludon(obs):\\n            return 18000, \\"Explorer: take Duraludon\\"\\n        if cid == RELICANTH and not has_in_play(obs, RELICANTH) and RELICANTH not in ids:\\n            return 15000, \\"Explorer: take Relicanth\\"\\n        sup_count = sum(1 for c in (my_state(obs).hand or []) if c and c.id in (EXPLORER, LILLIE))\\n        if cid in (EXPLORER, LILLIE) and sup_count == 0:\\n            return 12000, \\"Explorer: take supporter\\"\\n        return 0, \\"Explorer: let discard\\"\\n\\n    dura_ex_count = count_in_play(obs, DURALUDON) + count_in_play(obs, ARCHALUDON_EX)\\n    if cid == DURALUDON and DURALUDON not in ids and dura_ex_count <= 1:\\n        return 22000, \\"take Duraludon: backup\\"\\n    if cid == ARCHALUDON_EX and need_archaludon(obs):\\n        return 20000, \\"take Archaludon ex\\"\\n    if cid == DURALUDON and need_duraludon(obs):\\n        return 18000, \\"take Duraludon\\"\\n    if cid == CINDERACE:\\n        return -2000, \\"skip Cinderace\\"\\n    if cid == RELICANTH and not has_in_play(obs, RELICANTH):\\n        return 9000, \\"take Relicanth\\"\\n    if cid == METAL_ENERGY:\\n        return 8000, \\"take Metal Energy\\"\\n    if cid == EXPLORER and not obs.current.supporterPlayed:\\n        return 7500, \\"take Explorer\\"\\n    if cid == LILLIE and not obs.current.supporterPlayed:\\n        return 6500, \\"take Lillie\\"\\n    if cid == HERO_CAPE:\\n        has_target = any(p.id == ARCHALUDON_EX and not has_tool(p) for p in all_my_pokemon(obs))\\n        return (6000, \\"take Hero\'s Cape\\") if has_target else (1000, \\"generic take\\")\\n    if cid == FULL_METAL_LAB:\\n        return 5000, \\"take Full Metal Lab\\"\\n    if cid == BOSS:\\n        return 2500, \\"take Boss\\"\\n    return 1000, \\"generic take\\"\\n\\n\\ndef score_discard(obs, opt):\\n    card = option_card(obs, opt)\\n    cid = card.id if card else opt.cardId\\n    ids = hand_ids(obs)\\n    mt = metal_in_discard(obs)\\n    effect = getattr(obs.select, \\"effect\\", None)\\n    effect_id = effect.id if effect else None\\n\\n    if effect_id == ULTRA_BALL:\\n        mh = ids.count(METAL_ENERGY)\\n        if cid == METAL_ENERGY:\\n            if mt < 2 and mh >= 1:\\n                if getattr(opt, \'index\', None) == _first_option_index(obs, METAL_ENERGY):\\n                    return 20000, \\"UB: 1st Metal\\"\\n                return 8000, \\"UB: 2nd Metal\\"\\n            return 8000, \\"UB: Metal\\"\\n        if cid == CINDERACE:\\n            return (18000, \\"UB: Cinderace\\") if (mt >= 2 or mh == 0) else (14000, \\"UB: Cinderace\\")\\n        draw_count = ids.count(LILLIE) + ids.count(EXPLORER)\\n        if cid in (LILLIE, EXPLORER) and draw_count >= 2:\\n            return (12000 if cid == LILLIE else 11000), \\"UB: surplus supporter\\"\\n        if cid == ULTRA_BALL and ids.count(ULTRA_BALL) > 1:\\n            return 10000, \\"UB: duplicate\\"\\n        if cid in (LILLIE, EXPLORER) and draw_count <= 1:\\n            return -3000, \\"UB: keep last supporter\\"\\n\\n    if cid == METAL_ENERGY:\\n        if mt < 2:\\n            return 15000, \\"discard Metal\\"\\n        return (12000, \\"discard extra Metal\\") if ids.count(METAL_ENERGY) > 1 else (-1000, \\"keep last Metal\\")\\n    if cid == CINDERACE:\\n        return 10000, \\"discard Cinderace\\"\\n    if cid in {BOSS, FULL_METAL_LAB, POKEGEAR}:\\n        return 8500, \\"discard utility\\"\\n    if cid in {LILLIE, EXPLORER} and ids.count(cid) > 1:\\n        return 8000, \\"discard duplicate supporter\\"\\n    if cid == RELICANTH and (has_in_play(obs, RELICANTH) or ids.count(RELICANTH) > 1):\\n        return 6500, \\"discard extra Relicanth\\"\\n    if cid == ARCHALUDON_EX:\\n        return -5000, \\"keep Archaludon ex\\"\\n    if cid == DURALUDON:\\n        return -4000, \\"keep Duraludon\\"\\n    return 1000, \\"generic discard\\"\\n\\n\\ndef score_target(obs, opt):\\n    card = option_card(obs, opt)\\n    cid = card.id if card else opt.cardId\\n    ctx = obs.select.context\\n\\n    if ctx == SelectContext.ATTACH_TO:\\n        return (5000, \\"Metal\\") if cid == METAL_ENERGY else (1000, \\"attach\\")\\n\\n    if ctx == SelectContext.ATTACH_FROM:\\n        if card and energy_count(card) >= 3:\\n            return -5000, \\"skip: 3+ energy\\"\\n        if card and cid == CINDERACE and energy_count(card) >= 1:\\n            return -3000, \\"skip: Cinderace ready\\"\\n        return attach_target_score(obs, card, opt.area), \\"effect attach\\"\\n\\n    if ctx in {SelectContext.TO_FIELD, SelectContext.TO_BENCH}:\\n        if cid == ARCHALUDON_EX:\\n            return 18000, \\"target Archaludon ex\\"\\n        if cid == DURALUDON:\\n            return 16000, \\"target Duraludon\\"\\n        if cid == CINDERACE:\\n            return 3000, \\"avoid Cinderace\\"\\n\\n    if ctx == SelectContext.HEAL:\\n        return (20000 + damage_on(card), \\"heal Archaludon ex\\") if cid == ARCHALUDON_EX else (damage_on(card), \\"heal\\")\\n\\n    if ctx in {SelectContext.SWITCH, SelectContext.TO_ACTIVE}:\\n        yi = obs.current.yourIndex\\n        pi = getattr(opt, \'playerIndex\', yi)\\n        if pi != yi and card:\\n            # vs Hop: prioritize Snorlax (remove Extra Helpings)\\n            if detect_matchup(obs) == \\"hop\\" and cid == HOP_SNORLAX and card:\\n                active = active_pokemon(obs)\\n                e = energy_count(card)\\n                tools = len(getattr(card, \'tools\', None) or [])\\n                if active and active.id == CINDERACE:\\n                    # Cinderace: pull the least mobile Snorlax (low energy, no tools, high HP)\\n                    return 30000 - e * 100 - tools * 50 + card.hp, \\"Boss: Snorlax (immobile target)\\"\\n                else:\\n                    # Archaludon: pull the most threatening Snorlax (high energy, tools, high HP)\\n                    return 30000 + e * 100 + tools * 50 + card.hp, \\"Boss: Snorlax (biggest threat)\\"\\n            pv = prize_value(card)\\n            te = energy_count(card)\\n            killable = any(effective_damage(a[\\"damage\\"], card) >= card.hp\\n                           for a in planned_archaludon_attacks(obs))\\n            if killable:\\n                return 20000 + pv * 3000 + te * 100, \\"Boss: KO\\"\\n            return 5000 + pv * 1000 + te * 200, \\"Boss: drag\\"\\n        if cid == CINDERACE:\\n            return 16000, \\"promote Cinderace (retreat 0)\\"\\n        if cid == ARCHALUDON_EX:\\n            return 15000, \\"promote Archaludon ex\\"\\n        if cid == DURALUDON:\\n            return 8000, \\"promote Duraludon\\"\\n        return 1000, \\"generic promote\\"\\n\\n    if ctx == SelectContext.DAMAGE:\\n        hp = getattr(card, \\"hp\\", 999) if card else 999\\n        return 10000 - hp, \\"damage: lowest HP\\"\\n\\n    return 1000, \\"generic target\\"\\n\\n\\n# ── Choose & Agent ──\\n\\ndef choose_options(obs):\\n    scored = []\\n    for i, opt in enumerate(obs.select.option):\\n        try:\\n            score, reason = score_option(obs, opt)\\n        except Exception as e:\\n            score, reason = -999999, f\\"error {type(e).__name__}: {e}\\"\\n        scored.append((score, i, reason))\\n\\n    scored.sort(key=lambda x: (x[0], -x[1]), reverse=True)\\n\\n    selected = []\\n    for score, i, reason in scored:\\n        if len(selected) >= obs.select.maxCount:\\n            break\\n        if score < 0 and len(selected) >= obs.select.minCount:\\n            continue\\n        selected.append(i)\\n\\n    if len(selected) < obs.select.minCount:\\n        selected = [i for _, i, _ in scored[:obs.select.minCount]]\\n\\n    return selected\\n\\n\\ndef agent(obs_dict):\\n    obs = to_observation_class(obs_dict)\\n    if obs.select is None:\\n        global _opp_last_attack_id, _cur_turn_logs\\n        _opp_last_attack_id = None\\n        _cur_turn_logs.clear()\\n        return read_deck_csv()\\n    _update_opp_attack_tracking(obs)\\n    if not obs.select.option:\\n        return []\\n    try:\\n        return choose_options(obs)\\n    except Exception:\\n        return random.sample(list(range(len(obs.select.option))), obs.select.maxCount)\\n", "deck_csv": "8\\n8\\n8\\n8\\n8\\n8\\n8\\n8\\n8\\n8\\n8\\n57\\n169\\n169\\n169\\n169\\n190\\n190\\n190\\n190\\n666\\n666\\n666\\n666\\n1097\\n1097\\n1097\\n1121\\n1121\\n1121\\n1121\\n1122\\n1122\\n1122\\n1122\\n1147\\n1147\\n1147\\n1147\\n1152\\n1152\\n1152\\n1152\\n1159\\n1182\\n1182\\n1182\\n1185\\n1185\\n1185\\n1185\\n1213\\n1227\\n1227\\n1227\\n1227\\n1244\\n1244\\n1244\\n1244\\n", "main_hash": "a4c53101be301c181bd477204a72c0e5cba65fddd34d8cd0ec4d36e4b41c9518", "deck_hash": "fbe6ab59992260b0d6774abed19469be315521b5ed0546de8c20f329607693e6"}, "B": {"label": "B - Alakazam/Dunsparce Complement", "emoji": "🧠🎯", "build_name": "flex_alakazam_dunsparce_0000_seed", "archetype": "alakazam_dunsparce", "role": "Complementary challenger: different failure modes from metal tempo.", "short": "Best eligible Alakazam/Dunsparce complement after held-out debiasing.", "strategy": "Build hand size and board stability with the Alakazam line, Dudunsparce draw loops, and compact Psychic pressure.", "risk": "Less raw ceiling than A in the current panel, but useful as the decorrelated second portfolio option.", "selector_comment": "B: 🧠🎯 Alakazam/Dunsparce complement — decorrelated second upload candidate; provisional live gate.", "main_py": "import os\\nimport sys\\nfrom collections import defaultdict\\n\\nfrom cg.api import AreaType, CardType, EnergyType, Observation, SelectContext, OptionType, Card, Pokemon, all_card_data, to_observation_class\\n\\n\\"\\"\\"\\nAlakazam Deck\\nThis deck uses Alakazam\'s Powerful Hand attack (20 damage per card in hand)\\nwith a draw engine built around Kadabra/Alakazam Psychic Draw, Dudunsparce\'s\\nRun Away Draw, and Fezandipiti ex\'s Flip the Script.\\n\\"\\"\\"\\n\\n# Load deck.csv in the dataset\\nfile_path = \\"deck.csv\\"\\nif not os.path.exists(file_path):\\n    file_path = \\"/kaggle_simulations/agent/\\" + file_path\\nwith open(file_path, \\"r\\") as file:\\n    csv = file.read().split(\\"\\\\n\\")\\nmy_deck = []\\nfor i in range(60):\\n    my_deck.append(int(csv[i]))\\n\\n# Fetch card metadata database and create an ID-to-Card lookup table\\nall_card = all_card_data()\\ncard_table = {c.cardId: c for c in all_card}\\n\\n# Decklist\\nAbra = 741              # x4\\nKadabra = 742            # x4\\nAlakazam = 743           # x3\\nDunsparce = 305          # x3\\nDudunsparce = 66         # x2\\nFezandipiti_ex = 140     # x1\\nGenesect = 142           # x1\\nPsyduck = 858            # x1\\nShaymin = 343            # x1\\nRare_Candy = 1079        # x3\\nEnhanced_Hammer = 1081   # x3\\nBuddy_Buddy_Poffin = 1086  # x4\\nNight_Stretcher = 1097   # x1\\nSacred_Ash = 1129        # x1\\nPoke_Pad = 1152          # x4\\nLucky_Helmet = 1156      # x3\\nBoss_Orders = 1182       # x2\\nHilda = 1225             # x4\\nDawn = 1231              # x4\\nBattle_Cage = 1264       # x4\\nBasic_Psychic_Energy = 5   # x2\\nTelepath_Psychic_Energy = 19  # x4\\nEnriching_Energy = 13    # x1  (ACE SPEC)\\n\\n# Opponent card IDs to watch for\\nDuskull = 131\\nSlowpoke_IDs = (162, 327)\\nFroakie_IDs = (33, 945)\\nWellspring_Mask_Ogerpon_ex = 108\\nN_Darumaka = 257\\nDreepy = 119\\nDrakloak = 120\\nDragapult_ex = 121\\nMist_Energy = 11\\nRock_Fighting_Energy = 20\\n\\n# Attack IDs\\nATTACK_TELEPORTATION = 1070   # Abra: 10 dmg, cost {P}\\nATTACK_SUPER_PSY_BOLT = 1071  # Kadabra: 30 dmg, cost {P}\\nATTACK_POWERFUL_HAND = 1072   # Alakazam: 20 per card in hand, cost {P}\\n\\n# Card ID sets\\nABRA_LINE = {Abra, Kadabra, Alakazam}\\nDUNSPARCE_LINE = {Dunsparce, Dudunsparce}\\nPSYCHIC_ENERGY_IDS = {Basic_Psychic_Energy, Telepath_Psychic_Energy}\\n\\npre_turn = 0\\nability_used_dudunsparce = False\\nability_used_fezandipiti = False\\n\\n\\ndef get_card(obs: Observation, area: AreaType, index: int, player_index: int) -> Pokemon | Card | None:\\n    ps = obs.current.players[player_index]\\n    match area:\\n        case AreaType.DECK:\\n            return obs.select.deck[index]\\n        case AreaType.HAND:\\n            return ps.hand[index]\\n        case AreaType.DISCARD:\\n            return ps.discard[index]\\n        case AreaType.ACTIVE:\\n            return ps.active[index]\\n        case AreaType.BENCH:\\n            return ps.bench[index]\\n        case AreaType.PRIZE:\\n            return ps.prize[index]\\n        case AreaType.STADIUM:\\n            return obs.current.stadium[index]\\n        case AreaType.LOOKING:\\n            return obs.current.looking[index]\\n        case _:\\n            return None\\n\\n\\ndef prize_count(pokemon: Pokemon) -> int:\\n    data = card_table[pokemon.id]\\n    count = 3 if data.megaEx else 2 if data.ex else 1\\n    for card in pokemon.energyCards:\\n        if card.id == 12:  # Legacy Energy\\n            count -= 1\\n    for card in pokemon.tools:\\n        if card.id == 1172 and \\"Lillie\\" in data.name:\\n            count -= 1\\n    return max(0, count)\\n\\n\\ndef count_special_defense_energies(pokemon: Pokemon) -> int:\\n    cnt = 0\\n    for ec in pokemon.energyCards:\\n        if ec.id == Mist_Energy or ec.id == Rock_Fighting_Energy:\\n            cnt += 1\\n    return cnt\\n\\n\\n\\n# Crash-safety and submission diagnostics. The public policy below is preserved as\\n# _policy_agent(); the submitted agent() wrapper normalizes its option indices and\\n# falls back legally on every exception.\\n_DIAG = defaultdict(int)\\n\\n\\ndef diag_reset() -> None:\\n    _DIAG.clear()\\n\\n\\ndef diag_snapshot() -> dict:\\n    total = max(1, _DIAG.get(\\"decisions\\", 0))\\n    out = dict(_DIAG)\\n    out[\\"fallback_rate\\"] = (_DIAG.get(\\"policy_fallback\\", 0) + _DIAG.get(\\"obs_fallback\\", 0)) / total\\n    return out\\n\\n\\ndef _legal_fallback(select) -> list[int]:\\n    try:\\n        n = len(select.option)\\n        if n == 0 or select.maxCount <= 0:\\n            return []\\n        min_count = max(0, min(select.minCount, n))\\n        max_count = max(min_count, min(select.maxCount, n))\\n        return list(range(min_count if min_count > 0 else min(1, max_count)))\\n    except Exception:\\n        return []\\n\\n\\ndef _normalize_ordered_indices(indices, select) -> list[int]:\\n    try:\\n        n = len(select.option)\\n        if n == 0 or select.maxCount <= 0:\\n            return []\\n        min_count = max(0, min(select.minCount, n))\\n        max_count = max(min_count, min(select.maxCount, n))\\n        out = []\\n        seen = set()\\n        for idx in indices or []:\\n            if not isinstance(idx, int) or idx in seen or not (0 <= idx < n):\\n                continue\\n            seen.add(idx)\\n            out.append(idx)\\n            if len(out) >= max_count:\\n                break\\n        if len(out) < min_count:\\n            for idx in range(n):\\n                if idx not in seen:\\n                    out.append(idx)\\n                    seen.add(idx)\\n                if len(out) >= min_count:\\n                    break\\n        return out[:max_count]\\n    except Exception:\\n        return _legal_fallback(select)\\n\\n\\ndef _policy_agent(obs_dict: dict) -> list[int]:\\n    obs = to_observation_class(obs_dict)\\n    if obs.select is None:\\n        return my_deck\\n\\n    state = obs.current\\n    select = obs.select\\n    context = select.context\\n    my_index = state.yourIndex\\n    my_state = state.players[my_index]\\n    op_state = state.players[1 - my_index]\\n    my_prize_count = len(my_state.prize)\\n\\n    global pre_turn, ability_used_dudunsparce, ability_used_fezandipiti\\n    if pre_turn != state.turn:\\n        pre_turn = state.turn\\n        ability_used_dudunsparce = False\\n        ability_used_fezandipiti = False\\n\\n    # ---- Count cards on field / hand / discard ----\\n    field_counts = defaultdict(int)\\n    hand_counts = defaultdict(int)\\n    discard_counts = defaultdict(int)\\n\\n    my_field = []  # (field_index, pokemon) where 0=active, 1..=bench\\n    for card in my_state.active:\\n        if card is not None:\\n            field_counts[card.id] += 1\\n            my_field.append((0, card))\\n    for idx, card in enumerate(my_state.bench):\\n        if card is not None:\\n            field_counts[card.id] += 1\\n            my_field.append((idx + 1, card))\\n\\n    for card in my_state.hand:\\n        hand_counts[card.id] += 1\\n\\n    for card in my_state.discard:\\n        discard_counts[card.id] += 1\\n\\n    abra_line_on_field = field_counts[Abra] + field_counts[Kadabra] + field_counts[Alakazam]\\n    dunsparce_line_on_field = field_counts[Dunsparce] + field_counts[Dudunsparce]\\n\\n    # ---- Opponent field analysis ----\\n    op_all_pokemon = []\\n    for card in op_state.active:\\n        if card is not None:\\n            op_all_pokemon.append(card)\\n    for card in op_state.bench:\\n        if card is not None:\\n            op_all_pokemon.append(card)\\n\\n    op_has_duskull = any(p.id == Duskull for p in op_all_pokemon)\\n    op_has_water_threat = any(\\n        p.id in Slowpoke_IDs or p.id in Froakie_IDs\\n        or p.id == Wellspring_Mask_Ogerpon_ex or p.id == N_Darumaka\\n        for p in op_all_pokemon\\n    )\\n    op_has_dragapult_line = any(\\n        p.id in (Dreepy, Drakloak, Dragapult_ex) for p in op_all_pokemon\\n    )\\n\\n    # Detect if opponent has used ACE SPEC\\n    op_used_ace_spec = False\\n    for log in obs.logs:\\n        if hasattr(log, \'cardId\') and log.cardId is not None:\\n            cd = card_table.get(log.cardId)\\n            if cd and cd.aceSpec and hasattr(log, \'playerIndex\') and log.playerIndex == (1 - my_index):\\n                op_used_ace_spec = True\\n\\n    stadium_id = 0\\n    for card in state.stadium:\\n        stadium_id = card.id\\n\\n    bench_count = len(my_state.bench)\\n    bench_max = my_state.benchMax\\n    bench_free = bench_max - bench_count\\n\\n    # ---- Active pokemon info ----\\n    active_pokemon = my_state.active[0] if my_state.active else None\\n    active_id = active_pokemon.id if active_pokemon else -1\\n    active_has_psychic = False\\n    if active_pokemon:\\n        for ec in active_pokemon.energyCards:\\n            if ec.id in PSYCHIC_ENERGY_IDS:\\n                active_has_psychic = True\\n                break\\n\\n    # ---- Opponent active info ----\\n    op_active = op_state.active[0] if op_state.active else None\\n    op_active_hp = op_active.hp if op_active else 9999\\n\\n    # ---- Estimate Powerful Hand damage range ----\\n    hand_size = len(my_state.hand) if my_state.hand else my_state.handCount\\n\\n    def estimate_hand_increase():\\n        \\"\\"\\"Returns (min_increase, max_increase) of hand size this turn from draw effects.\\"\\"\\"\\n        min_inc = 0\\n        max_inc = 0\\n        for _, p in my_field:\\n            if p.id == Abra and hand_counts[Kadabra] > 0:\\n                max_inc += 1  # evolve Kadabra: hand -1, draw +2 = net +1\\n            elif p.id == Abra and hand_counts[Rare_Candy] > 0 and hand_counts[Alakazam] > 0:\\n                max_inc += 1  # Rare Candy + Alakazam: hand -2, draw +3 = net +1\\n            elif p.id == Kadabra and hand_counts[Alakazam] > 0:\\n                max_inc += 2  # evolve Alakazam: hand -1, draw +3 = net +2\\n            elif p.id == Dunsparce and hand_counts[Dudunsparce] > 0:\\n                max_inc += 1  # evolve: hand -1, ability draw +2 = net +1\\n            elif p.id == Dudunsparce:\\n                if not ability_used_dudunsparce:\\n                    max_inc += 3  # Run Away Draw\\n            elif p.id == Fezandipiti_ex:\\n                if not ability_used_fezandipiti:\\n                    max_inc += 3  # Flip the Script\\n        if hand_counts[Fezandipiti_ex] > 0 and bench_free > 0 and field_counts[Fezandipiti_ex] == 0:\\n            max_inc += 2  # play -1, ability +3 = net +2\\n\\n        # Supporter (only 1 can be used)\\n        supporter_options = []\\n        if not state.supporterPlayed:\\n            if hand_counts[Hilda] > 0:\\n                supporter_options.append(1)   # play -1, search +2 = net +1\\n            if hand_counts[Dawn] > 0:\\n                supporter_options.append(2)   # play -1, search +3 = net +2\\n            if hand_counts[Boss_Orders] > 0:\\n                supporter_options.append(-1)  # play -1 = net -1\\n        if supporter_options:\\n            max_inc += max(supporter_options)\\n\\n        # Enriching Energy attach: hand -1, draw +4 = net +3\\n        if hand_counts[Enriching_Energy] > 0 and not state.energyAttached:\\n            if active_id == Alakazam and active_has_psychic:\\n                max_inc += 3\\n        return min_inc, max_inc\\n\\n    min_hand_inc, max_hand_inc = estimate_hand_increase()\\n    max_hand_size = hand_size + max_hand_inc\\n    min_hand_size = hand_size + min_hand_inc\\n    max_damage = max_hand_size * 20\\n    min_damage = min_hand_size * 20\\n\\n    # ---- Target selection for attack ----\\n    target_idx = -1       # 0 = active, 1.. = bench\\n    target_pokemon = None\\n    target_use_boss = False\\n    target_can_kill = False\\n    target_prize_gain = 0\\n    target_hammer_needed = 0\\n    use_kadabra_finish = False\\n\\n    if state.turn >= 2 and op_active is not None:\\n        # Check Kadabra finisher: opponent active HP <= 30\\n        if op_active_hp <= 30 and (field_counts[Kadabra] >= 1 or active_id == Kadabra):\\n            target_idx = 0\\n            target_pokemon = op_active\\n            target_use_boss = False\\n            target_can_kill = True\\n            target_prize_gain = prize_count(op_active)\\n            use_kadabra_finish = True\\n        else:\\n            # Evaluate all opponent pokemon\\n            all_op = [(0, op_active)]\\n            for bi, bp in enumerate(op_state.bench):\\n                if bp is not None:\\n                    all_op.append((bi + 1, bp))\\n\\n            candidates = []\\n            for oi, pkmn in all_op:\\n                pz = prize_count(pkmn)\\n                sp_e = count_special_defense_energies(pkmn)\\n                eff_max_dmg = max_damage\\n                hm_need = 0\\n                if sp_e > 0:\\n                    if hand_counts[Enhanced_Hammer] >= sp_e:\\n                        hm_need = sp_e\\n                        eff_max_dmg = (max_hand_size - hm_need) * 20\\n                    else:\\n                        eff_max_dmg = 0\\n                ck = pkmn.hp <= eff_max_dmg and eff_max_dmg > 0\\n                candidates.append((oi, pkmn, pz, ck, hm_need))\\n\\n            # Priority 1: kill wins the game\\n            win_cands = [(oi, pk, pz, ck, hm) for oi, pk, pz, ck, hm in candidates if ck and my_prize_count <= pz]\\n            if win_cands:\\n                # Among winners, prefer active (no boss needed), then highest HP\\n                best = min(win_cands, key=lambda x: (0 if x[0] == 0 else 1, -x[1].hp))\\n                target_idx, target_pokemon, target_prize_gain, target_can_kill, target_hammer_needed = best\\n                target_use_boss = target_idx != 0\\n            else:\\n                # Priority 2: killable target with most prizes\\n                killable = [(oi, pk, pz, ck, hm) for oi, pk, pz, ck, hm in candidates if ck]\\n                if killable:\\n                    best = max(killable, key=lambda x: (x[2], x[1].hp))\\n                    target_idx, target_pokemon, target_prize_gain, target_can_kill, target_hammer_needed = best\\n                    target_use_boss = target_idx != 0\\n                else:\\n                    # Priority 3: just hit active\\n                    target_idx = 0\\n                    target_pokemon = op_active\\n                    target_use_boss = False\\n                    target_can_kill = False\\n                    target_prize_gain = 0\\n\\n    # Should we use Dudunsparce\'s ability?\\n    need_dudunsparce_draw = False\\n    if target_pokemon is not None and target_can_kill:\\n        needed = target_pokemon.hp\\n        current_dmg = (hand_size - target_hammer_needed) * 20\\n        if current_dmg < needed:\\n            need_dudunsparce_draw = True\\n\\n    # Do we need to attach energy to the active to retreat?\\n    need_retreat_energy = False\\n    if active_pokemon is not None and state.turn >= 2:\\n        active_is_attacker = (active_id == Alakazam and active_has_psychic) or (use_kadabra_finish and active_id == Kadabra)\\n        if not active_is_attacker:\\n            # Check if there\'s a better attacker on bench\\n            has_bench_attacker = False\\n            if use_kadabra_finish and field_counts[Kadabra] >= 1 and active_id != Kadabra:\\n                has_bench_attacker = True\\n            elif field_counts[Alakazam] >= 1 and active_id != Alakazam:\\n                has_bench_attacker = True\\n            elif field_counts[Kadabra] >= 1 and active_id != Kadabra:\\n                has_bench_attacker = True\\n            if has_bench_attacker:\\n                retreat_cost = card_table[active_pokemon.id].retreatCost\\n                active_energy_count = len(active_pokemon.energies)\\n                if active_energy_count < retreat_cost:\\n                    need_retreat_energy = True\\n\\n    # Do we need Fezandipiti ex\'s Flip the Script to kill the target?\\n    fez_hand_contribution = 0\\n    if field_counts[Fezandipiti_ex] >= 1 and not ability_used_fezandipiti:\\n        fez_hand_contribution = 3\\n    elif hand_counts[Fezandipiti_ex] > 0 and bench_free > 0 and field_counts[Fezandipiti_ex] == 0:\\n        fez_hand_contribution = 2  # play -1, ability +3 = net +2\\n    need_fezandipiti_draw = False\\n    if target_pokemon is not None and target_can_kill and fez_hand_contribution > 0:\\n        max_damage_without_fez = (max_hand_size - fez_hand_contribution - target_hammer_needed) * 20\\n        if max_damage_without_fez < target_pokemon.hp:\\n            need_fezandipiti_draw = True\\n\\n    # Also allow Fezandipiti if drawing could find key enablers (Boss, Rare Candy, Alakazam, Energy)\\n    need_fezandipiti_for_setup = False\\n    if target_pokemon is not None and target_can_kill and fez_hand_contribution > 0 and not need_fezandipiti_draw:\\n        # Missing Boss\'s Orders for bench target\\n        missing_boss = (target_use_boss and hand_counts[Boss_Orders] == 0\\n                        and not state.supporterPlayed)\\n        # Check if we have a ready attacker (Alakazam with psychic energy)\\n        has_ready_attacker = (active_id == Alakazam and active_has_psychic)\\n        if not has_ready_attacker:\\n            for _, p in my_field:\\n                if p.id == Alakazam and any(ec.id in PSYCHIC_ENERGY_IDS for ec in p.energyCards):\\n                    has_ready_attacker = True\\n                    break\\n        missing_attacker = False\\n        missing_energy = False\\n        if not has_ready_attacker:\\n            # Can we set up Alakazam this turn?\\n            can_evolve_to_alakazam = (field_counts[Kadabra] >= 1 and hand_counts[Alakazam] >= 1)\\n            can_rare_candy_alakazam = (field_counts[Abra] >= 1 and hand_counts[Rare_Candy] >= 1\\n                                       and hand_counts[Alakazam] >= 1)\\n            if not can_evolve_to_alakazam and not can_rare_candy_alakazam:\\n                # Missing evolution pieces\\n                if field_counts[Kadabra] >= 1 and hand_counts[Alakazam] == 0:\\n                    missing_attacker = True\\n                elif field_counts[Abra] >= 1 and (hand_counts[Rare_Candy] == 0 or hand_counts[Alakazam] == 0):\\n                    missing_attacker = True\\n            # Check if energy is available for the attacker\\n            energy_in_hand = (hand_counts[Basic_Psychic_Energy] + hand_counts[Telepath_Psychic_Energy]\\n                              + hand_counts[Enriching_Energy])\\n            if not state.energyAttached and energy_in_hand == 0:\\n                has_energized = any(\\n                    p.id in ABRA_LINE and any(ec.id in PSYCHIC_ENERGY_IDS for ec in p.energyCards)\\n                    for _, p in my_field\\n                )\\n                if not has_energized:\\n                    missing_energy = True\\n        if missing_boss or missing_attacker or missing_energy:\\n            need_fezandipiti_for_setup = True\\n\\n    # Deck safety: don\'t let deck count drop to <= prize count unless winning this turn\\n    can_win_this_turn = target_can_kill and my_prize_count <= target_prize_gain\\n    deck_count = my_state.deckCount\\n    # safe_draws: max cards we can draw from deck while keeping deck > prize count\\n    # We also need 1 card for the draw at start of next turn\\n    safe_draws = deck_count - my_prize_count - 1 if not can_win_this_turn else 999\\n\\n    # ---- Score each option ----\\n    scores = []\\n    for o in select.option:\\n        score = 0\\n\\n        if o.type == OptionType.NUMBER:\\n            score = o.number\\n\\n        elif o.type == OptionType.YES:\\n            score = 1\\n\\n        elif o.type == OptionType.CARD:\\n            card = get_card(obs, o.area, o.index, o.playerIndex)\\n            if card is None:\\n                scores.append(score)\\n                continue\\n            energy_count = len(card.energies) if isinstance(card, Pokemon) else 0\\n\\n            if context == SelectContext.SWITCH or context == SelectContext.TO_ACTIVE:\\n                if o.playerIndex == my_index:\\n                    if card.id == Alakazam:\\n                        score += 100 + energy_count * 10\\n                    elif card.id == Kadabra:\\n                        score += 90 if (op_active_hp <= 30) else 30\\n                    elif card.id == Abra:\\n                        score += 10\\n                    elif card.id in DUNSPARCE_LINE:\\n                        score += 5\\n                    else:\\n                        score += 1\\n                else:\\n                    if target_use_boss and target_pokemon is not None:\\n                        if o.index == target_idx - 1:\\n                            score += 100\\n\\n            elif context == SelectContext.SETUP_ACTIVE_POKEMON:\\n                if card.id == Abra:\\n                    score = 10\\n                elif card.id == Dunsparce:\\n                    score = 5\\n                elif card.id == Psyduck:\\n                    score = 2\\n                elif card.id == Shaymin:\\n                    score = 1\\n\\n            elif context == SelectContext.SETUP_BENCH_POKEMON:\\n                if card.id == Abra:\\n                    cur = field_counts[Abra] + field_counts[Kadabra] + field_counts[Alakazam]\\n                    score = 200 if cur == 0 else 100 + (3 - cur) * 10\\n                elif card.id == Dunsparce:\\n                    score = 150 if dunsparce_line_on_field == 0 else 50\\n\\n            elif context == SelectContext.TO_HAND:\\n                score = 200 - hand_counts.get(card.id, 0) * 50\\n                if card.id == Dudunsparce:\\n                    score += 80 if (field_counts[Dunsparce] >= 1 and field_counts[Dudunsparce] == 0) else -50\\n                elif card.id == Kadabra:\\n                    score += 70 if field_counts[Abra] >= 1 else -20\\n                elif card.id == Alakazam:\\n                    score += 60 if (field_counts[Kadabra] >= 1 or field_counts[Abra] >= 1) else -20\\n                elif card.id == Abra:\\n                    score += 50 if abra_line_on_field < 3 else -50\\n                elif card.id == Dunsparce:\\n                    score += 40 if dunsparce_line_on_field < 2 else -50\\n                elif card.id in PSYCHIC_ENERGY_IDS:\\n                    score += 30 if not state.energyAttached else -10\\n                elif card.id == Enriching_Energy:\\n                    score += 20\\n                elif card.id == Rare_Candy:\\n                    score += 40 if field_counts[Abra] >= 1 else -10\\n\\n            elif context == SelectContext.ATTACH_FROM:\\n                if isinstance(card, Pokemon):\\n                    if need_retreat_energy and o.area == AreaType.ACTIVE:\\n                        score = 150  # Must attach to active to retreat\\n                    elif len(card.energyCards) >= 1:\\n                        score = -1  # Don\'t attach 2+ energy to the same pokemon\\n                    elif card.id in ABRA_LINE:\\n                        score = 100\\n                        if card.id == Alakazam:\\n                            score += 20\\n                        elif card.id == Kadabra:\\n                            score += 10\\n                        if o.area == AreaType.ACTIVE:\\n                            score += 5\\n                    elif card.id in DUNSPARCE_LINE:\\n                        score = 50\\n                    else:\\n                        score = 10\\n\\n            elif context == SelectContext.TO_BENCH:\\n                if card.id == Abra:\\n                    score = 100\\n                elif card.id == Dunsparce:\\n                    score = 80\\n                elif card.id == Psyduck:\\n                    if op_has_duskull:\\n                        score = 60\\n                    else:\\n                        score = -1\\n                elif card.id == Shaymin:\\n                    if op_has_water_threat:\\n                        score = 40\\n                    else:\\n                        score = -1\\n\\n            elif context == SelectContext.TO_DECK:\\n                if card.id in ABRA_LINE:\\n                    score = 100\\n                elif card.id in DUNSPARCE_LINE:\\n                    score = 50\\n                else:\\n                    score = 10\\n\\n        elif o.type == OptionType.PLAY:\\n            card = get_card(obs, AreaType.HAND, o.index, my_index)\\n            data = card_table[card.id]\\n\\n            if data.cardType == CardType.POKEMON:\\n                score = 20000\\n                is_early = state.turn <= 2\\n\\n                if card.id == Abra:\\n                    if is_early:\\n                        score += 500\\n                    elif abra_line_on_field < 3:\\n                        score += 200\\n                    elif bench_free <= 1:\\n                        score = -1\\n                    else:\\n                        score += 50\\n\\n                elif card.id == Dunsparce:\\n                    if dunsparce_line_on_field < 1:\\n                        score += 400 if is_early else 100\\n                    elif dunsparce_line_on_field < 2:\\n                        score += 50\\n                    else:\\n                        score = -1\\n\\n                elif card.id == Fezandipiti_ex:\\n                    if need_fezandipiti_draw or need_fezandipiti_for_setup:\\n                        score += 80 if not is_early else 30\\n                    else:\\n                        score = -1  # Don\'t play unless Flip the Script is needed to kill\\n\\n                elif card.id == Genesect:\\n                    if not op_used_ace_spec and (hand_counts[Lucky_Helmet] > 0 or hand_counts[Poke_Pad] > 0):\\n                        score += 100\\n                    else:\\n                        score = -1\\n\\n                elif card.id == Psyduck:\\n                    if op_has_duskull:\\n                        score += 300\\n                    else:\\n                        score = -1\\n\\n                elif card.id == Shaymin:\\n                    if op_has_water_threat:\\n                        score += 300\\n                    else:\\n                        score = -1\\n\\n                # Keep at least 1 bench slot free\\n                if bench_free <= 1 and score > 0:\\n                    score -= 5000\\n\\n            else:\\n                score = 10000\\n\\n                if card.id == Buddy_Buddy_Poffin:\\n                    if safe_draws < 2:\\n                        score = -1  # Deck too thin (searches deck)\\n                    elif state.turn <= 2:\\n                        if abra_line_on_field < 3 or dunsparce_line_on_field < 1:\\n                            score = 18000\\n                        else:\\n                            score = 8000\\n                    else:\\n                        if abra_line_on_field < 3 or dunsparce_line_on_field < 2:\\n                            score = 15000\\n                        elif target_can_kill:\\n                            score = 8000\\n                        else:\\n                            score = -1\\n\\n                elif card.id == Poke_Pad:\\n                    if safe_draws < 1:\\n                        score = -1  # Deck too thin (searches deck)\\n                    elif state.turn <= 2:\\n                        score = 17000\\n                    else:\\n                        score = 14000 if abra_line_on_field < 3 else 12000\\n\\n                elif card.id == Rare_Candy:\\n                    if field_counts[Abra] >= 1 and hand_counts[Alakazam] >= 1 and safe_draws >= 3:\\n                        score = 16000\\n                    else:\\n                        score = -1\\n\\n                elif card.id == Night_Stretcher:\\n                    dis_abra = discard_counts[Abra] + discard_counts[Kadabra] + discard_counts[Alakazam]\\n                    if dis_abra >= 1:\\n                        score = 13000\\n                    elif discard_counts[Basic_Psychic_Energy] + discard_counts[Telepath_Psychic_Energy] >= 1:\\n                        score = 11000\\n                    else:\\n                        score = -1\\n\\n                elif card.id == Sacred_Ash:\\n                    dis_abra = discard_counts[Abra] + discard_counts[Kadabra] + discard_counts[Alakazam]\\n                    if dis_abra >= 2:\\n                        score = 13500\\n                    elif dis_abra >= 1:\\n                        score = 11000\\n                    else:\\n                        score = -1\\n\\n                elif card.id == Enhanced_Hammer:\\n                    if target_hammer_needed > 0:\\n                        score = 6500\\n                    else:\\n                        # Check if any opponent pokemon has special defense energy\\n                        any_special = any(count_special_defense_energies(p) > 0 for p in op_all_pokemon)\\n                        if any_special:\\n                            score = 5000\\n                        else:\\n                            score = -1\\n\\n                elif card.id == Lucky_Helmet:\\n                    score = 7000  # Will be handled via ATTACH\\n\\n                elif card.id == Boss_Orders:\\n                    if target_use_boss and target_can_kill:\\n                        score = 3200\\n                    else:\\n                        score = -1\\n\\n                elif card.id == Hilda:\\n                    if safe_draws >= 2:\\n                        score = 3000\\n                    else:\\n                        score = -1\\n\\n                elif card.id == Dawn:\\n                    if safe_draws >= 3:\\n                        score = 3100\\n                    else:\\n                        score = -1\\n\\n                elif card.id == Battle_Cage:\\n                    if op_has_dragapult_line:\\n                        score = 19000\\n                    elif stadium_id != 0:\\n                        score = 7000\\n                    else:\\n                        score = -1\\n\\n        elif o.type == OptionType.ATTACH:\\n            card = get_card(obs, AreaType.HAND, o.index, my_index)\\n            pokemon = get_card(obs, o.inPlayArea, o.inPlayIndex, my_index)\\n\\n            if card.id == Lucky_Helmet:\\n                score = 7000\\n                if pokemon.id == Genesect and not op_used_ace_spec:\\n                    score += 300\\n                elif o.inPlayArea == AreaType.ACTIVE:\\n                    score += 200\\n                else:\\n                    score += 50\\n\\n            elif card.id in PSYCHIC_ENERGY_IDS:\\n                if need_retreat_energy and o.inPlayArea == AreaType.ACTIVE:\\n                    score = 9500  # Must attach to active to retreat\\n                elif len(pokemon.energyCards) >= 1:\\n                    score = -1  # Don\'t attach 2+ energy to the same pokemon\\n                elif pokemon.id in ABRA_LINE:\\n                    score = 8000\\n                    if pokemon.id == Alakazam:\\n                        score += 30\\n                    elif pokemon.id == Kadabra:\\n                        score += 20\\n                    elif pokemon.id == Abra:\\n                        score += 10\\n                    if o.inPlayArea == AreaType.ACTIVE:\\n                        score += 5\\n                else:\\n                    score = -1\\n                # Telepath Psychic Energy searches 2 from deck\\n                if card.id == Telepath_Psychic_Energy and safe_draws < 2 and score > 0:\\n                    score = -1\\n\\n            elif card.id == Enriching_Energy:\\n                if need_retreat_energy and o.inPlayArea == AreaType.ACTIVE:\\n                    score = 9500  # Must attach to active to retreat\\n                elif len(pokemon.energyCards) >= 1:\\n                    score = -1  # Don\'t attach 2+ energy to the same pokemon\\n                elif pokemon.id in DUNSPARCE_LINE:\\n                    score = 8500\\n                    if pokemon.id == Dudunsparce:\\n                        score += 10\\n                else:\\n                    score = -1\\n                # Enriching Energy draws 4 from deck\\n                if card.id == Enriching_Energy and safe_draws < 4 and score > 0:\\n                    score = -1\\n\\n        elif o.type == OptionType.EVOLVE:\\n            card = get_card(obs, AreaType.HAND, o.index, my_index)\\n            pokemon = get_card(obs, o.inPlayArea, o.inPlayIndex, my_index)\\n            score = 9000\\n\\n            if card.id == Alakazam:\\n                if safe_draws < 3:\\n                    score = -1  # Deck too thin for Psychic Draw (3 cards)\\n                elif o.inPlayArea == AreaType.ACTIVE:\\n                    score += 200  # Active Alakazam = highest\\n                else:\\n                    score += 50  # Bench Alakazam\\n                score += len(pokemon.energies) * 10\\n\\n            elif card.id == Kadabra:\\n                if safe_draws < 2:\\n                    score = -1  # Deck too thin for Psychic Draw (2 cards)\\n                else:\\n                    score += 100\\n                    if len(pokemon.energies) == 0:\\n                        score += 50  # Evolve non-energy Abra first\\n                    else:\\n                        score -= 20\\n                        if hand_counts[Rare_Candy] > 0 and hand_counts[Alakazam] > 0:\\n                            score -= 100  # Save energy Abra for Rare Candy -> Alakazam\\n\\n            elif card.id == Dudunsparce:\\n                if safe_draws < 2:\\n                    score = -1  # Deck too thin for draw on evolve\\n                else:\\n                    score += 80\\n\\n        elif o.type == OptionType.ABILITY:\\n            card = get_card(obs, o.area, o.index, my_index)\\n            if card is None:\\n                scores.append(score)\\n                continue\\n\\n            if card.id == Dudunsparce:\\n                if need_dudunsparce_draw:\\n                    if safe_draws >= 3:\\n                        score = 30000\\n                    else:\\n                        score = -1  # Deck too thin\\n                else:\\n                    score = -1\\n            elif card.id == Fezandipiti_ex:\\n                if (need_fezandipiti_draw or need_fezandipiti_for_setup) and safe_draws >= 3:\\n                    score = 29000\\n                else:\\n                    score = -1  # Don\'t use unless needed to kill target\\n            elif card.id == Battle_Cage:\\n                score = 1\\n            else:\\n                score = 28000\\n\\n        elif o.type == OptionType.RETREAT:\\n            if active_id == Alakazam and active_has_psychic:\\n                score = -1\\n            elif use_kadabra_finish and active_id != Kadabra and field_counts[Kadabra] >= 1:\\n                score = 2500  # Retreat to bring Kadabra forward for finish\\n            elif active_id in (Abra, Dunsparce, Dudunsparce, Psyduck, Shaymin, Genesect):\\n                if field_counts[Alakazam] >= 1 or field_counts[Kadabra] >= 1:\\n                    score = 2000\\n                else:\\n                    score = -1\\n            else:\\n                score = -1\\n\\n        elif o.type == OptionType.ATTACK:\\n            score = 1000\\n            if o.attackId == ATTACK_POWERFUL_HAND:\\n                score += 500\\n            elif o.attackId == ATTACK_SUPER_PSY_BOLT:\\n                if op_active_hp <= 30:\\n                    score += 600  # Kadabra finisher\\n                else:\\n                    score += 100\\n            elif o.attackId == ATTACK_TELEPORTATION:\\n                score += 50\\n\\n        scores.append(score)\\n\\n    # Select in descending order of score\\n    desc_indices = [i for i, _ in sorted(enumerate(scores), key=lambda x: x[1], reverse=True)]\\n\\n    if context == SelectContext.MAIN:\\n        o = select.option[desc_indices[0]]\\n        if o.type == OptionType.ABILITY:\\n            card = get_card(obs, o.area, o.index, my_index)\\n            if card is not None:\\n                if card.id == Dudunsparce:\\n                    ability_used_dudunsparce = True\\n                elif card.id == Fezandipiti_ex:\\n                    ability_used_fezandipiti = True\\n\\n    return desc_indices[:select.maxCount]\\n\\n\\ndef agent(obs_dict: dict) -> list[int]:\\n    try:\\n        if isinstance(obs_dict, dict) and obs_dict.get(\\"select\\") is None:\\n            _DIAG[\\"deck_returns\\"] += 1\\n            return my_deck\\n    except Exception:\\n        pass\\n\\n    _DIAG[\\"decisions\\"] += 1\\n    try:\\n        obs = to_observation_class(obs_dict)\\n        if obs.select is None:\\n            _DIAG[\\"deck_returns\\"] += 1\\n            _DIAG[\\"decisions\\"] -= 1\\n            return my_deck\\n        raw = _policy_agent(obs_dict)\\n        selection = _normalize_ordered_indices(raw, obs.select)\\n        if selection:\\n            _DIAG[\\"policy_ok\\"] += 1\\n            return selection\\n        _DIAG[\\"policy_fallback\\"] += 1\\n        return _legal_fallback(obs.select)\\n    except Exception:\\n        _DIAG[\\"policy_fallback\\"] += 1\\n        try:\\n            obs = to_observation_class(obs_dict)\\n            if obs.select is None:\\n                _DIAG[\\"deck_returns\\"] += 1\\n                _DIAG[\\"decisions\\"] -= 1\\n                return my_deck\\n            return _legal_fallback(obs.select)\\n        except Exception:\\n            _DIAG[\\"obs_fallback\\"] += 1\\n            if isinstance(obs_dict, dict) and obs_dict.get(\\"select\\") is None:\\n                return my_deck\\n            return []\\n", "deck_csv": "5\\n5\\n5\\n13\\n19\\n19\\n19\\n19\\n66\\n66\\n66\\n305\\n305\\n305\\n305\\n741\\n741\\n741\\n741\\n742\\n742\\n742\\n742\\n743\\n743\\n743\\n743\\n1079\\n1079\\n1079\\n1079\\n1081\\n1081\\n1081\\n1081\\n1086\\n1086\\n1086\\n1086\\n1097\\n1097\\n1097\\n1129\\n1152\\n1152\\n1152\\n1152\\n1182\\n1182\\n1182\\n1184\\n1225\\n1225\\n1225\\n1225\\n1231\\n1231\\n1231\\n1231\\n1264\\n", "main_hash": "46aae79654eca7d91e9a3c840d92e38d3ac6271b052379df43dc630163f68225", "deck_hash": "0f8fb632ade2833645af8c6ffe2b282fe24e7446ee8b23f3e468b8410d8ea36c"}}')
CARD_NAMES = json.loads('{"5": "Basic Psychic Energy", "8": "Basic Metal Energy", "13": "Enriching Energy", "19": "Telepath Psychic Energy", "57": "Relicanth", "66": "Dudunsparce", "169": "Duraludon", "190": "Archaludon ex", "305": "Dunsparce", "666": "Cinderace", "741": "Abra", "742": "Kadabra", "743": "Alakazam", "1079": "Rare Candy", "1081": "Enhanced Hammer", "1086": "Buddy-Buddy Poffin", "1097": "Night Stretcher", "1121": "Ultra Ball", "1122": "Pokegear 3.0", "1129": "Sacred Ash", "1147": "Jumbo Ice Cream", "1152": "Poke Pad", "1159": "Hero\'s Cape", "1182": "Boss\'s Orders", "1185": "Explorer\'s Guidance", "1213": "Flex card 1213", "1225": "Hilda", "1227": "Lillie\'s Determination", "1231": "Dawn", "1244": "Full Metal Lab", "1264": "Battle Cage"}')
CG_API_PY = 'from dataclasses import dataclass\nfrom enum import IntEnum\nimport json\nimport ctypes\n\nfrom .sim import lib\nfrom .utils import to_dataclass, json_to_dataclass\n\n#region Enums\n\nclass AreaType(IntEnum):\n    DECK = 1,\n    HAND = 2,\n    DISCARD = 3, # Discard Pile\n    ACTIVE = 4, # Active Spot\n    BENCH = 5,\n    PRIZE = 6,\n    STADIUM = 7,\n    ENERGY = 8,\n    TOOL = 9,\n    PRE_EVOLUTION = 10, # The pre-evolved form of the Pokémon in play.\n    PLAYER = 11,\n    LOOKING = 12, # The card you are looking.\n\nclass EnergyType(IntEnum):\n    COLORLESS = 0,\n    GRASS = 1,\n    FIRE = 2,\n    WATER = 3,\n    LIGHTNING = 4,\n    PSYCHIC = 5,\n    FIGHTING = 6,\n    DARKNESS = 7,\n    METAL = 8,\n    DRAGON = 9,\n    RAINBOW = 10, # Every Types\n    TEAM_ROCKET = 11, # PSYCHIC and DARKNESS \n\nclass CardType(IntEnum):\n    POKEMON = 0,\n    ITEM = 1,\n    TOOL = 2, # Pokémon Tool\n    SUPPORTER = 3,\n    STADIUM = 4,\n    BASIC_ENERGY = 5,\n    SPECIAL_ENERGY = 6,\n\nclass SpecialConditionType(IntEnum):\n    POISON = 0,\n    BURN = 1,\n    SLEEP = 2,\n    PARALYZE = 3,\n    CONFUSE = 4,\n\nclass SelectType(IntEnum):\n    MAIN = 0, # OptionType: PLAY, ATTACH, EVOLVE, ABILITY, DISCARD, RETREAT, ATTACK, END\n    CARD = 1, # OptionType: CARD\n    ATTACHED_CARD = 2, # OptionType: TOOL_CARD, ENERGY_CARD\n    CARD_OR_ATTACHED_CARD = 3, # OptionType: CARD, TOOL_CARD, ENERGY_CARD\n    ENERGY = 4, # OptionType: ENERGY\n    SKILL = 5, # OptionType: SKILL\n    ATTACK = 6, # OptionType: ATTACK\n    EVOLVE = 7, # OptionType: EVOLVE\n    COUNT = 8, # OptionType: NUMBER\n    YES_NO = 9, # OptionType: YES, NO\n    SPECIAL_CONDITION = 10, # OptionType: SPECIAL_CONDITION\n    \nclass SelectContext(IntEnum):\n    MAIN = 0, # Main. Main selection.\n    SETUP_ACTIVE_POKEMON = 1, # Card. Select the Pokémon to put into your Active Spot during Set Up.\n    SETUP_BENCH_POKEMON = 2, # Card. Select the Pokémon to put onto your Bench during Set Up.\n    SWITCH = 3, # Card. Select the Pokémon to swap with the one in your Active Spot.\n    TO_ACTIVE = 4, # Card. Select the Pokémon to put into your Active Spot.\n    TO_BENCH = 5, # Card. Select the Pokémon to put onto your Bench.\n    TO_FIELD = 6, # Card. Select the Pokémon to put into play.\n    TO_HAND = 7, # Card. Select the card to add to your hand.\n    DISCARD = 8, # Card. Select the card to discard.\n    TO_DECK = 9, # Card. Select the card to return to your deck.\n    TO_DECK_BOTTOM = 10, # Card. Select the card to return to the bottom of your deck.\n    TO_PRIZE = 11, # Card. Select the card to add to your prize.\n    NOT_MOVE = 12, # Card. Select the card to remain where it is.\n    DAMAGE_COUNTER = 13, # Card. Select the Pokémon to place damage counters on.\n    DAMAGE_COUNTER_ANY = 14, # Card. Select the Pokémon to place damage counters on using the effect that lets you place them as you like.\n    DAMAGE = 15, # Card. Select the Pokémon to deal damage.\n    REMOVE_DAMAGE_COUNTER = 16, # Card. Select the Pokémon to remove damage counters from.\n    HEAL = 17, # Card. Select the Pokémon to heal.\n    EVOLVES_FROM = 18, # Card. Select the Pokémon to evolve from.\n    EVOLVES_TO = 19, # Card. Select the Pokémon to evolve into.\n    DEVOLVE = 20, # Card. Select the Pokémon to devolve.\n    ATTACH_FROM = 21, # Card. Select the Pokémon to attach the card to.\n    ATTACH_TO = 22, # Card. Select the card to attach to the Pokémon.\n    DETACH_FROM = 23, # Card. Select the Pokémon to remove the card from.\n    LOOK = 24, # Card. Select the card to look at.\n    EFFECT_TARGET = 25, # Card. Select the card to apply the effect to.\n    DISCARD_ENERGY_CARD = 26, # AttachedCard. Select the Energy card to discard.\n    DISCARD_TOOL_CARD = 27, # AttachedCard. Select the Pokémon tool to trash.\n    SWITCH_ENERGY_CARD = 28, # AttachedCard. Select the energy card to replace.\n    DISCARD_CARD_OR_ATTACHED_CARD = 29, # CardOrAttachedCard. Select the card to discard.\n    DISCARD_ENERGY = 30, # Energy. Select the energy to discard.\n    TO_HAND_ENERGY = 31, # Energy. Select the energy to return to your hand.\n    TO_DECK_ENERGY = 32, # Energy. Select the energy to return to the deck.\n    SWITCH_ENERGY = 33, # Energy. Select the energy to switch.\n    SKILL_ORDER = 34, # Skill. Select the order of effect activation.\n    ATTACK = 35, # Attack. Select the Attack to use.\n    DISABLE_ATTACK = 36, # Attack. Select the Attack to disable.\n    EVOLVE = 37, # Evolve. Select the Pokémon that is the evolution source and the Pokémon that is the evolution target.\n    DRAW_COUNT = 38, # Count. Select how many cards to draw.\n    DAMAGE_COUNTER_COUNT = 39, # Count. Select how many damage counters to place.\n    REMOVE_DAMAGE_COUNTER_COUNT = 40, # Count. Select how many damage counters to remove.\n    IS_FIRST = 41, # YesNo. Would you like to go first?\n    MULLIGAN = 42, # YesNo. Would you like to redraw the cards?\n    ACTIVATE = 43, # YesNo. Would you like to activate the effect?\n    FIRST_EFFECT = 44, # YesNo. Would you like to select the first effect?\n    MORE_DEVOLVE = 45, # YesNo. Do you want to devolve it further?\n    COIN_HEAD = 46, # YesNo. Do you want to choose heads?\n    AFFECT_SPECIAL_CONDITION = 47, # SpecialCondition. Choose the special condition to affect.\n    RECOVER_SPECIAL_CONDITION = 48, # SpecialCondition. Choose the special condition to recover.\n    # Please note that new elements may be appended to the Enum during the competition.\n\nclass OptionType(IntEnum):\n    # number (int):Count.\n    NUMBER = 0, # Number to select.\n\n    YES = 1, # Select Yes.\n\n    NO = 2, # Select No.\n\n    # area (AreaType):Area where the card is located.\n    # index (int):Index within the area.\n    # playerIndex (int):The owning player of the card.\n    CARD = 3, # Card to select.\n\n    # area (AreaType):Area of the attached Pokémon.\n    # index (int):Index within the area of the attached Pokémon.\n    # playerIndex (int):The owning player of the Pokémon.\n    # toolIndex (int):Index within the tool.\n    TOOL_CARD = 4, # Pokémon Tool Card to select.\n\n    # area (AreaType):Area of the attached Pokémon.\n    # index (int):Index within the area of the attached Pokémon.\n    # playerIndex (int):The owning player of the Pokémon.\n    # energyIndex (int):Index within the energy card.\n    ENERGY_CARD = 5, # Energy Card to select.\n\n    # area (AreaType):Area of the attached Pokémon.\n    # index (int):Index within the area of the attached Pokémon.\n    # playerIndex (int):The owning player of the Pokémon.\n    # energyIndex (int):Index within the energy card.\n    # count (int):How many energy units does it correspond to?\n    ENERGY = 6, # Energy to select.\n\n    # index (int):Index within the hand.\n    PLAY = 7, # Play a card from your hand.\n\n    # area (AreaType):Area of the card to attach.\n    # index (int):Index within the area of the card to attach.\n    # inPlayArea (AreaType):Area of the Pokémon on the field.\n    # inPlayIndex (int):Index within the area of the Pokémon on the field.\n    ATTACH = 8, # Attach a card to a Pokémon.\n\n    # area (AreaType):Area of the evolved card.\n    # index (int):Index within the area of the evolved card.\n    # inPlayArea (AreaType):Area of the Pokémon on the field.\n    # inPlayIndex (int):Index within the area of the Pokémon on the field.\n    EVOLVE = 9, # Select an Evolution.\n\n    # area (AreaType):Area where the card is located.\n    # index (int):Index within the area.\n    ABILITY = 10, # Use an Ability.\n\n    # area (AreaType):Area where the card is located.\n    # index (int):Index within the area.\n    DISCARD = 11, # Discard a card in play.\n\n    RETREAT = 12, # Retreat Active Pokémon.\n\n    # attackId (int):Attack ID\n    ATTACK = 13, # Select an Attack.\n\n    END = 14, # Turn End.\n\n    # cardId (int):Card ID. When the Card ID is 0, it means handling a Special Condition.\n    # serial (int):Card serial\n    SKILL = 15, # Select the order of card skills.\n\n    # specialConditionType (SpecialConditionType):Special Condition Type\n    SPECIAL_CONDITION = 16, # Select the Special Condition.\n\nclass LogType(IntEnum):\n    # playerIndex (int)\n    SHUFFLE = 0, # Shuffle deck.\n\n    # playerIndex (int)\n    # hasBasicPokemon (bool):If false, then no Basic Pokémon exist.\n    HAS_BASIC_POKEMON = 1,\n\n    # playerIndex (int)\n    TURN_START = 2, # Start turn.\n\n    # playerIndex (int)\n    TURN_END = 3, # End turn.\n\n    # playerIndex (int)\n    # cardId (int):Drawn card ID\n    # serial (int):Drawn card serial\n    DRAW = 4, # Drew a card from deck.\n\n    # playerIndex (int)\n    DRAW_REVERSE = 5, # Your opponent drew a card from their deck.\n\n    # playerIndex (int)\n    # cardId (int):Moved card. ID\n    # serial (int):Moved card. serial\n    # fromArea (AreaType):Area before movement.\n    # toArea (AreaType):Area after movement.\n    MOVE_CARD = 6, # A card moved.\n\n    # playerIndex (int)\n    # fromArea (AreaType):Area before movement.\n    # toArea (AreaType):Area after movement.\n    MOVE_CARD_REVERSE = 7, # A card moved face-down.\n\n    # playerIndex (int)\n    # cardIdActive (int):Moving to the Bench Pokémon ID\n    # serialActive (int):Moving to the Bench Pokémon serial\n    # cardIdBench (int):Moving to the Active Pokémon ID\n    # serialBench (int):Moving to the Active Pokémon serial\n    SWITCH = 8, # Pokémon were switched.\n\n    # playerIndex (int)\n    # cardIdBefore (int):Pokémon before change. ID\n    # serialBefore (int):Pokémon before change. serial\n    # cardIdAfter (int):Pokémon after change. ID\n    # serialAfter (int):Pokémon after change. serial\n    CHANGE = 9, # Change the Pokémon.\n\n    # playerIndex (int)\n    # cardId (int):Played card ID\n    # serial (int):Played card serial\n    PLAY = 10, # Played a card from hand.\n\n    # playerIndex (int)\n    # cardId (int):Attached card ID\n    # serial (int):Attached card serial\n    # cardIdTarget (int):Pokémon card ID\n    # serialTarget (int):Pokémon card serial\n    ATTACH = 11, # Attached a card to a Pokémon.\n\n    # playerIndex (int)\n    # cardId (int):Evolved card ID\n    # serial (int):Evolved card serial\n    # cardIdTarget (int):Pokémon card ID\n    # serialTarget (int):Pokémon card serial\n    EVOLVE = 12, # Evolved a Pokémon.\n\n    # playerIndex (int)\n    # cardId (int):Devolved card ID\n    # serial (int):Devolved card serial\n    # cardIdTarget (int):Pokémon card ID\n    # serialTarget (int):Pokémon card serial\n    DEVOLVE = 13, # Devolved a Pokémon.\n\n    # playerIndex (int)\n    # cardId (int):Attached card ID\n    # serial (int):Attached card serial\n    # cardIdBefore (int):Pokémon that were attached with cards. ID\n    # serialBefore (int):Pokémon that were attached with cards. serial\n    # cardIdAfter (int):Pokémon that were newly attached with cards. ID\n    # serialAfter (int):Pokémon that were newly attached with cards. serial\n    MOVE_ATTACHED = 14, # Move the attached card.\n\n    # playerIndex (int)\n    # cardId (int):Pokémon that use attack. ID\n    # serial (int):Pokémon that use attack. serial\n    # attackId (int):Attack ID\n    ATTACK = 15, # Pokémon Attack.\n\n    # playerIndex (int)\n    # cardId (int):HP changed card ID\n    # serial (int):HP changed card serial\n    # value (int):Amount of change.\n    # putDamageCounter (bool):True if the HP change is due to the effect of placing a damage counter.\n    HP_CHANGE = 16, # A Pokémon’s HP changed.\n\n    # playerIndex (int)\n    # isRecover (bool):If true, the special condition has been recovered.\n    # cardId (int): ID\n    # serial (int): serial\n    POISONED = 17, # Poisoned.\n\n    # playerIndex (int)\n    # isRecover (bool):If true, the special condition has been recovered.\n    # cardId (int): ID\n    # serial (int): serial\n    BURNED = 18, # Burned.\n\n    # playerIndex (int)\n    # isRecover (bool):If true, the special condition has been recovered.\n    # cardId (int): ID\n    # serial (int): serial\n    ASLEEP = 19, # Fell asleep.\n\n    # playerIndex (int)\n    # isRecover (bool):If true, the special condition has been recovered.\n    # cardId (int): ID\n    # serial (int): serial\n    PARALYZED = 20, # Paralyzed.\n\n    # playerIndex (int)\n    # isRecover (bool):If true, the special condition has been recovered.\n    # cardId (int): ID\n    # serial (int): serial\n    CONFUSED = 21, # Confused.\n\n    # playerIndex (int)\n    # head (bool):True if coin is head.\n    COIN = 22, # Result of the coin flip.\n\n    # result (int):If 0, the player with player index 0 wins; if 1, the player with player index 1 wins; if 2, it\'s a draw.\n    # reason (int):1: 0 Prize cards. 2: Start turn with 0 deck cards. 3: No Pokémon in Active Spot. 4: A card effect.\n    RESULT = 23, # Result of the match.\n    \n    # Please note that new elements may be appended to the Enum during the competition.\n\n#endregion Enums\n\n\n# Please note that new attributes may be appended to each class during the competition.\n\n#region Observation class\n\n@dataclass\nclass Card:\n    id: int  # CardData ID.\n    serial: int  # Serial Number: A unique value assigned to each card in the match.\n    playerIndex: int  # Represents which player\'s card.\n\n@dataclass\nclass Pokemon:\n    id: int  # CardData ID.\n    serial: int  # Serial Number: A unique value assigned to each card in the match.\n    hp: int  # Current HP.\n    maxHp: int  # Current Max HP.\n    appearThisTurn: bool  # True if played this turn.\n    energies: list[EnergyType]  # Energies Array\n    energyCards: list[Card]  # Attached Energy Card Array\n    tools: list[Card]  # Attached Pokémon Tool Array\n    preEvolution: list[Card]  # Pre-evolution Card Array\n \n@dataclass\nclass PlayerState:\n    active: list[Pokemon | None]  # Active Pokémon (None if the card is facedown). The array size is either 0 or 1.\n    bench: list[Pokemon]  # Bench Pokémon.\n    benchMax: int  # Maximum Bench Count.\n    deckCount: int  # Remaining Cards in Deck.\n    discard: list[Card]  # Discard pile Card Array.\n    prize: list[Card | None]  # Prize cards (None if the card is facedown). The first element is the bottom of the prize, and the last element is the top.\n    handCount: int  # Number of Cards in Hand.\n    hand: list[Card] | None  # Hand Card Array. None for the opponent.\n    poisoned: bool # Active Pokémon is Poisoned.\n    burned: bool # Active Pokémon is Burned.\n    asleep: bool # Active Pokémon is Asleep.\n    paralyzed: bool # Active Pokémon is Paralyzed.\n    confused: bool # Active Pokémon is Confused.\n\n@dataclass\nclass State:\n    turn: int  # Turn Count: 1 indicates the first turn for the starting player. 2 indicates the first turn for the second player. 3 indicates the second turn for the starting player. 0 denotes a time before the starting player\'s first turn.\n    turnActionCount: int  # Number of Actions Taken This Turn.\n    yourIndex: int  # Which player is making the selection? (Your Player Index.) 0 or 1.\n    firstPlayer: int  # Starting Player Index. When the starting player has not been determined, the value is -1.\n    supporterPlayed: bool  # True if a supporter has already been used this turn.\n    stadiumPlayed: bool  # True if a stadium has already been used this turn.\n    energyAttached: bool  # True if the manual Energy attachment for this turn has already been used.\n    retreated: bool  # True if retreated this turn.\n    result: int # Win player index. -1 if not battle finished.\n    stadium: list[Card]  # Stadium Card. The array size is either 0 or 1.\n    looking: list[Card | None] | None  # Looking cards (None if the card is facedown). None if not looking cards.\n    players: list[PlayerState]  # An array of player states. The number of elements is 2.\n\n@dataclass\nclass Option:\n    type: OptionType  # Use this parameter to determine which option it is.\n    number: int | None = None\n    area: AreaType | None = None\n    index: int | None = None\n    playerIndex: int | None = None\n    toolIndex: int | None = None\n    energyIndex: int | None = None\n    count: int | None = None\n    inPlayArea: AreaType | None = None\n    inPlayIndex: int | None = None\n    attackId: int | None = None\n    cardId: int | None = None\n    serial: int | None = None\n    specialConditionType: SpecialConditionType | None = None\n\n@dataclass\nclass SelectData:\n    type: SelectType  # Selection type.\n    context: SelectContext  # What is being selected?\n    minCount: int  # Minimum number of selections. It can also be 0.\n    maxCount: int  # Maximum number of selections. Never exceeds len(option).\n    remainDamageCounter: int  # Remaining number of damage counters that can be placed.\n    remainEnergyCost: int  # Used when the type is Energy. The remaining required energy count.\n    option: list[Option]  # Array of options.\n    deck: list[Card] | None  # An array of cards; None unless selecting cards from the deck.\n    contextCard: Card | None  # Which card is the selection concerning? This is sent when the context is "Activate"; otherwise, it is null.\n    effect: Card | None  # The card that is activating the effect currently being processed.\n    \n@dataclass\nclass Log:\n    type: LogType  # Use this parameter to determine which log it is.\n    playerIndex: int | None = None\n    hasBasicPokemon: bool | None = None\n    cardId: int | None = None\n    serial: int | None = None\n    fromArea: AreaType | None = None\n    toArea: AreaType | None = None\n    cardIdActive: int | None = None\n    serialActive: int | None = None\n    cardIdBench: int | None = None\n    serialBench: int | None = None\n    cardIdBefore: int | None = None\n    serialBefore: int | None = None\n    cardIdAfter: int | None = None\n    serialAfter: int | None = None\n    cardIdTarget: int | None = None\n    serialTarget: int | None = None\n    attackId: int | None = None\n    value: int | None = None\n    putDamageCounter: bool | None = None\n    isRecover: bool | None = None\n    head: bool | None = None\n    result: int | None = None\n    reason: int | None = None\n    \n@dataclass\nclass Observation:\n    select: SelectData | None  # Selection information. At the time of the initial deck selection, it will be None.\n    logs: list[Log]  # Events that have occurred since the last selection.\n    current: State | None  # Current state. At the time of the initial deck selection, it will be None.\n    search_begin_input: str | None = None # Input to the search_begin function.\n\n#endregion Observation class\n\n@dataclass\nclass SearchState:\n    observation: Observation  # New observation. search_begin_input is None.\n    searchId: int  #  Search state ID.\n    \n@dataclass\nclass ApiResult:\n    state: SearchState | None # Search state.\n    error: int # Error if not 0.\n\n# Abilities and effects at the time of card play.\n@dataclass\nclass Skill:\n    name: str  # Skill name.\n    text: str  # Explanation.\n\n@dataclass\nclass CardData:\n    cardId: int  # Card ID.\n    name: str  # Card name.\n    cardType: CardType  # Card type\n    retreatCost: int  # Energy cost required to retreat.\n    hp: int  # Pokémon HP.\n    weakness: EnergyType | None  # Pokémon weakness.\n    resistance: EnergyType | None  # Pokémon resistance.\n    energyType: EnergyType  # Pokémon or Basic Energy type.\n    basic: bool # True if Basic Pokémon.\n    stage1: bool # True if Stage1 Pokémon.\n    stage2: bool # True if Stage2 Pokémon.\n    ex: bool # True if Pokémon ex(include Mega Evolution Pokémon ex). When your Pokémon ex is Knocked Out, your opponent takes 2 prize cards(exclude Mega Evolution Pokémon ex).\n    megaEx: bool # True if Mega Evolution Pokémon ex. When your Mega Evolution Pokémon ex is Knocked Out, your opponent takes 3 prize cards.\n    tera: bool  # True if Tera Pokémon. Tera Pokémon take no damage from attacks as long as they are on the Bench.\n    aceSpec: bool  # True if ACE SPEC. You can\'t have more than 1 ACE SPEC card in your deck.\n    evolvesFrom: str | None  # If the Pokémon has evolved, then the name of its pre-evolution. Otherwise, None.\n    skills: list[Skill]  # The skills that the card has.\n    attacks: list[int]  # IDs of usable attacks.\n\n@dataclass\nclass Attack:\n    attackId: int  # Attack ID.\n    name: str  # Attack name.\n    text: str  # Explanation.\n    damage: int  # Attack damage\n    energies: list[EnergyType]  # Energy required to use.\n    \n\n#region functions\n\ndef all_card_data() -> list[CardData]:\n    """Return all cards."""\n    bs = lib.AllCard()\n    js = bs.decode()\n    cards = json.loads(js)\n    return [to_dataclass(v, CardData) for v in cards]\n\ndef all_attack() -> list[Attack]:\n    """Return all attacks."""\n    bs = lib.AllAttack()\n    js = bs.decode()\n    cards = json.loads(js)\n    return [to_dataclass(v, Attack) for v in cards]\n\ndef to_observation_class(obs: dict) -> Observation:\n    """dict to Observation class.\n\n    Returns:\n        Observation: Observation dataclass instance.\n    """\n    return to_dataclass(obs, Observation)\n\ndef search_begin(agent_observation: Observation,\n                 your_deck: list[int],\n                 your_prize: list[int],\n                 opponent_deck: list[int],\n                 opponent_prize: list[int],\n                 opponent_hand: list[int],\n                 opponent_active: list[int],\n                 manual_coin: bool = False\n    ) -> SearchState:\n    """Begin search.\n\n    Args:\n        agent_observation: You must input the observation argument passed to your agent function exactly as is.\n        your_deck: Predicted Card ID your Deck. It must have the same number of cards as your deck. If Observation.select.deck != None, ignored this.\n        your_prize: Predicted Card ID your Prize cards. It must have the same number of cards as your prize.\n        opponent_deck: Predicted Card ID opponent\'s deck. It must have the same number of cards as opponent\'s deck. At setup, at least one Basic Pokémon card is required.\n        opponent_prize: Predicted Card ID opponent\'s prize cards. It must have the same number of cards as opponent\'s prize.\n        opponent_hand: Predicted Card ID opponent\'s hand. It must have the same number of cards as opponent\'s hand.\n        opponent_active: Predicted Card ID opponent\'s Active Pokémon. Only if there is a face-down Pokémon in your opponent’s Active Spot. This ID must be a Pokémon card ID.\n        manual_coin: If True, the coin\'s heads or tails can be chosen.\n\n    Returns:\n        SearchState: Root search state.\n    """\n    global agent_ptr\n    \n    if "agent_ptr" not in globals():\n        agent_ptr = lib.AgentStart()\n    \n    sbi = agent_observation.search_begin_input\n    if sbi == None:\n        raise ValueError("Not agent observation.")\n\n    state = agent_observation.current\n    your_index = state.yourIndex\n\n    if agent_observation.select.deck != None:\n        your_deck = []\n    elif len(your_deck) < state.players[your_index].deckCount:\n        raise ValueError("your_deck does not match the number of cards in your deck.")\n    \n    if len(your_prize) < len(state.players[your_index].prize):\n        raise ValueError("your_prize does not match the number of cards in your prize.")\n    elif len(opponent_deck) < state.players[1 - your_index].deckCount:\n        raise ValueError("opponent_deck does not match the number of cards in opponent\'s deck.")\n    elif len(opponent_prize) < len(state.players[1 - your_index].prize):\n        raise ValueError("opponent_prize does not match the number of cards in opponent\'s prize.")\n    elif len(opponent_hand) < state.players[1 - your_index].handCount:\n        raise ValueError("opponent_hand does not match the number of cards in opponent\'s hand.")\n    \n    active = state.players[1 - your_index].active\n    if len(active) > 0 and active[0] == None:\n        if len(opponent_active) == 0:\n            raise ValueError("You need to predict the opponent\'s Active Pokémon.")\n    else:\n        opponent_active = []\n    \n    bs = lib.SearchBegin(agent_ptr,\n                         sbi.encode("ascii"),\n                         len(sbi),\n                         (ctypes.c_int*len(your_deck))(*your_deck),\n                         (ctypes.c_int*len(your_prize))(*your_prize),\n                         (ctypes.c_int*len(opponent_deck))(*opponent_deck),\n                         (ctypes.c_int*len(opponent_prize))(*opponent_prize),\n                         (ctypes.c_int*len(opponent_hand))(*opponent_hand),\n                         (ctypes.c_int*len(opponent_active))(*opponent_active),\n                         int(manual_coin))\n    result = json_to_dataclass(bs, ApiResult)\n    if result.error != 0:\n        if result.error == 1:\n            raise ValueError("Invalid Card ID.")\n        elif result.error == 2:\n            raise ValueError("Active card must be the ID of a Pokémon card.")\n        elif result.error == 30:\n            raise ValueError("agent_ptr broken.")\n        else:\n            raise RuntimeError()\n\n    return result.state\n\ndef search_step(search_id: int, select: list[int]) -> SearchState:\n    """Proceed to the next selection.\n    \n    Args:\n        search_id: Search ID.\n        select: Chosen option index.\n\n    Returns:\n        SearchSate: State for the next selection.\n    """\n    bs = lib.SearchStep(agent_ptr, search_id, (ctypes.c_int*len(select))(*select), len(select))\n    result = json_to_dataclass(bs, ApiResult)\n    if result.error != 0:\n        if result.error == 1:\n            raise ValueError("There is no element with the specified search_id.")\n        elif result.error == 2:\n            raise ValueError("Released item.")\n        elif result.error == 3:\n            raise ValueError("Cannot be selected because the battle has ended.")\n        elif result.error == 4:\n            raise ValueError("Must be Observation.select.minCount <= len(select) <= Observation.select.maxCount.")\n        elif result.error == 5:\n            raise ValueError("Must be 0 <= select elements < len(Observation.select.option).")\n        elif result.error == 6:\n            raise ValueError("Duplicate select elements.")\n        elif result.error == 30:\n            raise ValueError("agent_ptr broken.")\n        else:\n            raise RuntimeError()\n    \n    return result.state\n\ndef search_end() -> None:\n    """Terminate the search. Memory used during the search will be reused in the next search."""\n    lib.SearchEnd(agent_ptr)\n\ndef search_release(search_id: int) -> None:\n    """Delete the state with the specified ID and make the memory available for reuse.\n    \n    Args:\n        search_id: Search ID.\n    """\n    lib.SearchRelease(agent_ptr, search_id)\n\n#endregion functions\n\n'
CG_UTILS_PY = 'import json\n\n\ndef to_dataclass(dic: dict, cls: type):\n    """\n    Convert a dictionary to a dataclass instance recursively.\n\n    This function matches keys in the dictionary to the fields of the given dataclass `cls`,\n    and recursively constructs nested dataclass instances if needed.\n\n    Args:\n        dic (dict): The source dictionary.\n        cls (type): The target dataclass type.\n\n    Returns:\n        Any: An instance of the target dataclass, or None if input is None.\n    """\n    if dic is None:\n        return None\n\n    field_types = {f.name: f.type for f in cls.__dataclass_fields__.values()}\n    d = {}\n\n    for key, value in dic.items():\n        if key in field_types:\n            if isinstance(value, dict):\n                c = field_types[key]\n                if hasattr(c, "__args__"):\n                    c = c.__args__[0]\n                d[key] = to_dataclass(value, c)\n            elif isinstance(value, list):\n                c = field_types[key].__args__[0]\n                if hasattr(c, "__args__"):\n                    c = c.__args__[0]\n                    if hasattr(c, "__args__"):\n                        c = c.__args__[0]\n                if not hasattr(c, "__dataclass_fields__"):\n                    d[key] = value\n                else:\n                    d[key] = [to_dataclass(v, c) for v in value]\n            else:\n                d[key] = value\n\n    return cls(**d)\n\n\ndef json_to_dataclass(bs: bytes, cls: type):\n    """\n    Convert a JSON byte string to a dataclass instance.\n\n    This function decodes the JSON bytes and uses `to_dataclass()` to\n    recursively convert the resulting dictionary to a dataclass instance.\n\n    Args:\n        bs (bytes): JSON data in bytes.\n        cls (type): The target dataclass type.\n\n    Returns:\n        Any: An instance of the target dataclass.\n    """\n    js = bs.decode()\n    dic = json.loads(js)\n    return to_dataclass(dic, cls)\n'
PALETTE = {'archaludon':'#5B8DEF','alakazam_dunsparce':'#8E6AD8','lucario':'#E7A83E','hop_trevenant':'#4B9B6E','starmie':'#2EA8C9','dragapult':'#D75F7A','good':'#38A169','warn':'#DD6B20','bad':'#E53E3E','neutral':'#718096'}

def read_csv_text(key):
    txt = DATA.get(key, '')
    return pd.read_csv(StringIO(txt)) if txt else pd.DataFrame()
field_df = read_csv_text('field_chart_csv')
trend_df = read_csv_text('trend_chart_csv')
ev_df = read_csv_text('ev_chart_csv')
screen_df = read_csv_text('screen_candidates_csv')
holdout_df = read_csv_text('holdout_candidates_csv')
debias_df = read_csv_text('debias_csv')
gap_df = read_csv_text('gap_csv')
hold_cand_df = read_csv_text('hold_candidates_csv')
pair_df = read_csv_text('pair_scores_csv')

def clean_name(x):
    s = str(x).replace('_', ' ').replace('other:', '').replace('team rocket', 'TR')
    return ' '.join(w.capitalize() if w.upper() != 'TR' else 'TR' for w in s.split())
def short_name(x):
    s = str(x); low = s.lower()
    mapping = {'alakazam_dunsparce':'Alakazam','hop_trevenant':'Hop/Trevenant','iono_kilowattrel':'Iono/KW','festival_thwackey':'Thwackey','team_rocket_spidops':'TR Spidops'}
    if s in mapping: return mapping[s]
    if 'honchkrow' in low or 'murkrow' in low: return 'TR Honchkrow'
    if 'okidogi' in low and 'solrock' in low: return 'Okidogi/Solrock'
    if 'cynthia' in low or 'gible' in low: return 'Cynthia/Gible'
    if 'ogerpon' in low: return 'Ogerpon'
    return clean_name(s)[:22]
def pct(x, digits=1):
    return '' if pd.isna(x) else f'{100*float(x):.{digits}f}%'
def short_candidate(x):
    s = str(x)
    mapping = {
        'flex_archaludon_0018_minus1182_plus1213': 'A Archaludon 0018',
        'flex_alakazam_dunsparce_0000_seed': 'B Alakazam 0000',
        'flex_archaludon_0001_minus1122_plus8': 'Archaludon 0001',
        'flex_alakazam_dunsparce_0002_minus5_plus1266': 'Alakazam 0002',
        'public_alakazam_5th': 'Public Alakazam',
        'meta_snapshot_930_submission_b': '930 Lucario',
        'public1084_lucario': '1084 Lucario',
        'public915_search_lucario': 'Search Lucario',
    }
    return mapping.get(s, s.replace('flex_', '').replace('_', ' ')[:24])
def nice_table(df, cols=None, percent_cols=(), rename=None, rows=12):
    d = df.copy()
    if cols: d = d[[c for c in cols if c in d.columns]]
    if rows: d = d.head(rows)
    for c in percent_cols:
        if c in d.columns: d[c] = d[c].map(lambda v: pct(v))
    if rename: d = d.rename(columns=rename)
    display(d)
def card_count_table(selection):
    payload = AGENT_PAYLOADS[selection]
    counts = pd.Series([int(x) for x in payload['deck_csv'].strip().splitlines()]).value_counts().sort_values(ascending=False)
    out = pd.DataFrame({'card_id': counts.index.astype(str), 'copies': counts.values})
    out['card'] = out['card_id'].map(lambda x: CARD_NAMES.get(x, 'Card ' + x))
    return out[['card','card_id','copies']]
def profile_cards():
    rows = []
    for key, payload in AGENT_PAYLOADS.items():
        hc = hold_cand_df[hold_cand_df['candidate_id'].eq(payload['build_name'])]
        gg = gap_df[gap_df['candidate_id'].eq(payload['build_name'])]
        h = hc.iloc[0].to_dict() if len(hc) else {}
        g = gg.iloc[0].to_dict() if len(gg) else {}
        rows.append(f'''
        <div style="border:1px solid #d8dee9;border-radius:10px;padding:14px;margin:10px 0;background:#fbfdff;">
          <div style="font-size:18px;font-weight:700;">{payload['emoji']} {key} · {payload['label']}</div>
          <div style="color:#4a5568;margin-top:4px;">{payload['role']}</div>
          <div style="margin-top:8px;"><b>Strategy:</b> {payload['strategy']}</div>
          <div style="margin-top:8px;"><b>Evidence:</b> held score <code>{float(h.get('held_score', float('nan'))):.3f}</code>, weighted score <code>{float(h.get('weighted_score_rate', float('nan'))):.3f}</code>, edge <code>{float(h.get('weighted_edge_vs_field', float('nan'))):+.3f}</code>, coverage <code>{float(h.get('covered_field_weight', float('nan'))):.3f}</code>.</div>
          <div style="margin-top:8px;"><b>Gate:</b> candidate decision <code>{h.get('decision','')}</code>, gap clean <code>{g.get('gap_clean','')}</code>, live-reference status <code>{g.get('live_reference_gate_decision','')}</code>.</div>
          <div style="margin-top:8px;color:#744210;"><b>Risk:</b> {payload['risk']}</div>
        </div>
        ''')
    return HTML('\n'.join(rows))
def plot_field_shape():
    df = field_df.head(10).copy(); df['label'] = df['archetype'].map(short_name)
    fig, ax = plt.subplots(figsize=(11.2, 6.4), constrained_layout=True)
    sizes = np.clip(df['games'] / df['games'].max() * 950, 100, 950)
    colors = [PALETTE.get(a, '#A0AEC0') for a in df['archetype']]
    ax.scatter(df['usage_pct'], df['score_pct'], s=sizes, c=colors, alpha=0.82, edgecolor='white', linewidth=1.2)
    ax.axhline(50, color='#718096', ls='--', lw=1); ax.axvline(10, color='#CBD5E0', ls=':', lw=1)
    for _, r in df.iterrows(): ax.text(r['usage_pct']+0.25, r['score_pct']+0.35, r['label'], fontsize=8.5)
    ax.set_title('Field share vs score rate: pressure is not just popularity'); ax.set_xlabel('usage share (%)'); ax.set_ylabel('score rate (%)')
    ax.set_xlim(0, max(22, df['usage_pct'].max()+3)); ax.set_ylim(min(30, df['score_pct'].min()-5), max(70, df['score_pct'].max()+5))
    ax.text(0.5, 51.2, '50% score baseline', color='#4A5568', fontsize=8)
    plt.show()
def plot_meta_movement():
    df = trend_df.head(10).sort_values('usage_delta_pct')
    fig, ax = plt.subplots(figsize=(11.2, 5.8), constrained_layout=True); colors = ['#38A169' if v >= 0 else '#E53E3E' for v in df['usage_delta_pct']]
    ax.barh(df['archetype'].map(short_name), df['usage_delta_pct'], color=colors, alpha=0.9); ax.axvline(0, color='#718096', lw=1)
    for y, v in enumerate(df['usage_delta_pct']): ax.text(v + (0.12 if v >= 0 else -0.12), y, f'{v:+.1f} pp', va='center', ha='left' if v >= 0 else 'right', fontsize=8)
    ax.set_title('Rising and fading pressure'); ax.set_xlabel('usage-share change, percentage points')
    plt.show()
def plot_candidate_benchmark():
    df = hold_cand_df.copy().sort_values('held_score', ascending=True)
    labels = [short_candidate(c) for c in df['candidate_id']]
    fig, ax = plt.subplots(figsize=(12.8, 7.2)); colors = ['#38A169' if d == 'PROMOTE_CANDIDATE' else '#A0AEC0' for d in df['decision']]
    ax.barh(labels, df['held_score'], color=colors, alpha=0.9); ax.axvline(0.60, color='#718096', ls='--', lw=1)
    for y, v in enumerate(df['held_score']): ax.text(v + 0.008, y, f'{v:.3f}', va='center', fontsize=8)
    ax.set_title('Hold-period candidate score after coverage, gap, and debias gates'); ax.set_xlabel('held score'); ax.set_xlim(0.24, max(0.80, df['held_score'].max()+0.05))
    fig.subplots_adjust(left=0.30, right=0.97, top=0.90, bottom=0.11)
    plt.show()
def plot_pair_scores():
    df = pair_df.copy().sort_values('pair_held_score', ascending=True).tail(10)
    labels = [f"{short_candidate(a)} + {short_candidate(b)}" for a,b in zip(df['candidate_a'], df['candidate_b'])]
    fig, ax = plt.subplots(figsize=(12.4, 6.2)); colors = ['#38A169' if d == 'PROMOTE_PAIR' else '#CBD5E0' for d in df['decision']]
    y_pos = np.arange(len(df))
    ax.barh(y_pos, df['pair_held_score'], color=colors, alpha=0.9)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels)
    for y, v in enumerate(df['pair_held_score']): ax.text(v + 0.006, y, f'{v:.3f}', va='center', fontsize=8)
    ax.set_title('Portfolio score: eligible pair beats same-archetype duplicates'); ax.set_xlabel('pair held score'); ax.set_xlim(0.60, max(0.82, df['pair_held_score'].max()+0.04))
    fig.subplots_adjust(left=0.34, right=0.97, top=0.90, bottom=0.11)
    plt.show()


In [ ]:
plot_field_shape()
nice_table(field_df[['archetype','games','usage_share','score_rate','wilson_low']].head(10), percent_cols=['usage_share','score_rate','wilson_low'], rename={'usage_share':'usage','score_rate':'score','wilson_low':'score_wilson_low'})

In [ ]:
plot_meta_movement()
nice_table(trend_df[['archetype','usage_delta','usage_0627','score_0627','games_0627']].head(10), percent_cols=['usage_delta','usage_0627','score_0627'], rename={'usage_delta':'usage_delta','usage_0627':'usage','score_0627':'score'})

In [ ]:
plot_candidate_benchmark()
cols = ['candidate_id','candidate_archetype','weighted_score_rate','weighted_edge_vs_field','covered_field_weight','held_score','gap_clean','debias_status','decision']
nice_table(hold_cand_df.sort_values('held_score', ascending=False), cols=cols, percent_cols=['weighted_score_rate','weighted_edge_vs_field','covered_field_weight','held_score'], rows=12)

In [ ]:
plot_pair_scores()
nice_table(pair_df.sort_values('pair_held_score', ascending=False), cols=['candidate_a','candidate_b','candidate_a_archetype','candidate_b_archetype','mean_best_score_rate','min_member_held_score','unique_edge_a','unique_edge_b','pair_held_score','decision','reason'], percent_cols=['mean_best_score_rate','min_member_held_score','pair_held_score'], rows=10)

In [ ]:
display(profile_cards())
for key in ['A','B']:
    print(f'\n{key} deck signature: {AGENT_PAYLOADS[key]["label"]}')
    nice_table(card_count_table(key), rows=18)

In [ ]:
gate_rows = gap_df[['candidate_id','candidate_archetype','local_weighted_score','local_weighted_edge','covered_field_weight','live_reference_gate_decision','gap_clean','dominant_causes','recommended_route']]
nice_table(gate_rows.sort_values(['gap_clean','local_weighted_score'], ascending=[False,False]), percent_cols=['local_weighted_score','local_weighted_edge','covered_field_weight'], rows=12)
print('Final decision counts:', DATA['promotion_counts'])
print('Decision mode:', DATA['decision_mode'])

In [ ]:
AGENT_SELECTION = "A"

In [ ]:
display(profile_cards())

In [ ]:
selection = str(AGENT_SELECTION).strip().upper()
if selection not in AGENT_PAYLOADS:
    raise ValueError(f'Unknown AGENT_SELECTION={selection!r}; choose one of {sorted(AGENT_PAYLOADS)}')
payload = AGENT_PAYLOADS[selection]
work = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
build_dir = work / 'selected_agent_build'
if build_dir.exists(): shutil.rmtree(build_dir)
build_dir.mkdir(parents=True, exist_ok=True)
(build_dir / 'main.py').write_text(payload['main_py'], encoding='utf-8')
(build_dir / 'deck.csv').write_text(payload['deck_csv'], encoding='utf-8')
py_compile.compile(str(build_dir / 'main.py'), doraise=True)
def find_cg_source():
    candidates = [Path('/kaggle/input/competitions/pokemon-tcg-ai-battle/sample_submission/sample_submission/cg'), Path('/kaggle/input/pokemon-tcg-ai-battle/sample_submission/sample_submission/cg'), Path('/kaggle/input/competitions/pokemon-tcg-ai-battle/sample_submission/cg'), Path('/kaggle/input/pokemon-tcg-ai-battle/sample_submission/cg')]
    for c in candidates:
        if c.exists() and (c / 'sim.py').exists() and (c / 'game.py').exists(): return c
    input_root = Path('/kaggle/input')
    if input_root.exists():
        for sim_file in sorted(input_root.rglob('cg/sim.py')):
            c = sim_file.parent
            if (c / 'game.py').exists(): return c
    spec = importlib.util.find_spec('kaggle_environments')
    if spec and spec.submodule_search_locations:
        pkg = Path(list(spec.submodule_search_locations)[0]) / 'envs' / 'cabt' / 'cg'
        if pkg.exists() and (pkg / 'sim.py').exists() and (pkg / 'game.py').exists(): return pkg
    return None
cg_source = find_cg_source()
if cg_source is None:
    raise FileNotFoundError('Could not locate the CABT cg engine package from competition input or kaggle_environments.')
shutil.copytree(cg_source, build_dir / 'cg', ignore=shutil.ignore_patterns('__pycache__', '*.pyc', '*.pyo'))
if not (build_dir / 'cg' / 'api.py').exists(): (build_dir / 'cg' / 'api.py').write_text(CG_API_PY, encoding='utf-8')
if not (build_dir / 'cg' / 'utils.py').exists(): (build_dir / 'cg' / 'utils.py').write_text(CG_UTILS_PY, encoding='utf-8')
if platform.system().lower() != 'darwin':
    old_cwd = Path.cwd(); sys.path.insert(0, str(build_dir)); os.chdir(build_dir)
    try:
        spec = importlib.util.spec_from_file_location('selected_main', build_dir / 'main.py')
        mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod); assert callable(mod.agent)
    finally:
        os.chdir(old_cwd)
        if str(build_dir) in sys.path: sys.path.remove(str(build_dir))
archive_path = work / 'submission.tar.gz'
if archive_path.exists(): archive_path.unlink()
with tarfile.open(archive_path, 'w:gz') as tar:
    tar.add(build_dir / 'main.py', arcname='main.py'); tar.add(build_dir / 'deck.csv', arcname='deck.csv'); tar.add(build_dir / 'cg', arcname='cg')
summary = pd.DataFrame([{'selected': selection, 'profile': payload['label'], 'decision_mode': DATA['decision_mode'], 'archive': archive_path.name, 'main_sha256': payload['main_hash'][:16], 'deck_sha256': payload['deck_hash'][:16]}])
display(summary)
print(f'Built submission.tar.gz for profile {selection}: {payload["label"]}')
